## Analysis of Kaggle Energy and Weather Dataset (Kolasniwash)

Here I will analyse an energy dataset provided by Kolasniwash on Kaggle on the electrical demand, energy generation, prices, and weather in Spain (see, <https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather/data>). The dataset consists of two data files, energy_dataset.csv and weather_features.csv, containing the energy demand, generation, and prices along with weather data, respectively. The aim will be to gain meaningful insights into energy demand, generation, and prices as a function of the weather and try to predict future energy demand, generation, and prices by building predictive machine learning models.

The general steps will be the following:
1. Import basic Python packages needed for the data analysis.
2. Read in and store the data into memory using Pandas dataframes.
3. Perform some basic data exploration:
    * Length of data, number of columns, column data types
    * Number of missing values in each column
    * Median, mean, minimum, maximum, and ranges of each of the relevant column
    * Cadence of the data (time intervals)
    * Plots of the data as a function of time
    * Look for outliers
    * Look for interesting trends
4. Data cleaning:
    * Replace missing values (such as using the median, a rolling mean, spline interpolatation, or local regression depending on the column)
    * Replace outliers and invalid values (using methods listed above)
5. If there are any categorical data, apply ordinal encoding (for ordered values) and one-hot encoding (for values with no ordered relationship) to it to convert it to numerical data.
6. Merge the energy and weather data into a single dataframe as a function of time. If the cadence and timing of the energy and weather data are different, then interpolation might need to be used to have both datasets on the same time scale.
7. Check for correlations between energy demand, generation, and prices and the weather features using Pearson and Spearman correlation heatmap matrices.
8. Engineer new feature that will increase the predictive power of ML models such as:
    * Day of week, where Monday is 0 and Sunday is 6 or use one-hot encoding
    * Work day and weekend, where work days (Monday through Friday) are set to 0 and weekends to 1 (or one-hot encode this)
    * Solar flux as a function of time and city. Solar energy generation is highly dependent on the amount of solar energy received (and energy demand is too)
    * Local time as a fraction of the day so that we can use the time as a model feature
    * Sunrise and sunset as fractions of a day
    * Convert local time, time of week, sunrise, and sunset to cyclical times using sine and cosine for periodicity as this might help the model to better understand the time-based repeating nature of energy usage and generation more easily
9. Trial different machine learning models:
    * Random Forest Regression
    * XGBoost
    * Time-series specific models such as ARIMA, LSTM, or Prophet
10. Compare the perform of the different ML models. Also think about additional features that could be engineered and retest.
11. Put data on an AWS and develop DS pipeline to perform above tasks.

### Step One: Import Python Libraries

In [ ]:
# Import the essential Python libraries we will definitely be using for this analysis.
import numpy as np
import pandas as pd
import os
import csv
from datetime import datetime          # To help deal with time series data.
import math
from tqdm import tqdm                  # Progress bar for loops that take a while.

# Import visualization libraries, I'm a big fan of Bokeh, but will also make use of Matplotlib.
# Matplotlib plotting library and functions.
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns

# Bokeh plotting library and functions. Note, this is probably an overkill but I have used most of these in my other projects.
from bokeh.io import output_notebook, show
from bokeh.models.annotations.labels import Label
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Whisker, BoxAnnotation, Arrow, OpenHead, Span
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import Legend, LinearAxis, Range1d, ColumnDataSource, LabelSet, HoverTool, DatetimeTickFormatter

### Step Two: Read the data from the two files into Pandas Dataframes

I will use Pandas to read in the data and store it in dataframes given the mixed data types. I will use the read_csv function in Pandas to read in these csv files.

In [ ]:
# Get current directory. I will use this to construct the file path to the data files.
current_dir = str(os.getcwd())

files_dir = current_dir + '/data/'

# dataframe holding power usage/generation data. Use default options unless we encounter issues.
power_pd = pd.read_csv(files_dir+'energy_dataset.csv', header=0)

# dataframe holding weather info. Use default options unless we encounter issues.
weather_pd = pd.read_csv(files_dir+'weather_features.csv', header=0)

### Step Three: Data exploration

Now I will explore these datasets to determine data types in the columns, length of the datasets, number of missing values, and basic statistics (mean, median, standard deviation, minimum and maximum values, etc.)

### Power Data Exploration

In [ ]:
# Print the length (number of rows) of the Power dataset.
print('Number of Rows in Power dataset: ', len(power_pd))

# Print the data type of the columns.
print('Columns and data types in the Power dataset: \n', power_pd.dtypes)

# Print the first five rows of power dataset to get a sense of the format of the data.
power_pd.head()

In [ ]:
# Basic summary of power dataset using describe function from Pandas for the columns that are type float or integer.
power_pd.describe()

In [ ]:
# Check on the percentage of missing values in each column. We will need to do some data wrangling (e.g., deal with data/time columns) 
# and cleaning (e.g., deal with missing values).
# Use isnull() and sum() function to count missing values in all the columns.
print("Percentage of missing entries for each column:\n", 100.0*(power_pd.isnull().sum()/len(power_pd)))

In [ ]:
# Check on the percentage of zero values in each column. Some columns may only contain 0 values so will need to be
# removed.
print("Percentage of zero entries for each column:\n", 100.0*((power_pd == 0).sum()/len(power_pd)))

Based on the columns in the power dataset, we can see it contains a few different parts:
* A time column, given as YYYY-MM-DD HH:MM:SS+HH:MM, where +HH:MM is the UTC to Central European Time (CET).
* Several energy generation columns for the different types of energy sources including:
    - generation biomass
    - generation fossil brown coal/lignite
    - generation fossil coal-derived gas
    - generation fossil gas
    - generation fossil hard coal
    - generation fossil oil
    - generation fossil oil shale
    - generation fossil peat
    - generation geothermal
    - generation hydro pumped storage aggregated
    - generation hydro pumped storage consumption
    - generation hydro run-of-river and poundage
    - generation hydro water reservoir
    - generation marine
    - generation nuclear
    - generation other
    - generation other renewable
    - generation solar
    - generation waste
    - generation wind offshore
    - generation wind onshore
* Three energy generation forecast columns that includes:
    - forecast solar day ahead
    - forecast wind offshore eday ahead
    - forecast wind onshore day ahead
* Two total power load columns, actual and forecast:
    - total load forecast
    - total load actual
* Two price columns, actual and day ahead:
    - price day ahead
    - price actual

Looking ahead, the power forecast columns will need to removed and put into a new dataframe as we don't want to use them in our ML model (other than to compare the performance of our ML model to the forecast model by TSO). Similarily, the total load forecast and price day ahead should be moved to a seperate dataframe.

Almost all columns contain at least some null or zero values, however, some columns contain entirely null or zero values and these should be removed. These columns include:
* generation fossil coal-derived gas
* generation fossil oil shale
* generation fossil peat
* generation geothermal
* generation hydro pumped storage aggregated
* generation marine
* generation wind offshore
* forecast wind offshore eday ahead

### Weather Data Exploration

In [ ]:
# Length of weather dataset.
print('Number of Rows in weather dataset: ', len(weather_pd))

# Print the data type of the columns.
print('Columns and data types in the weather dataset: \n', weather_pd.dtypes)

# Print the first five rows of weather dataset to get a sense of the format of the data.
weather_pd.head()

In [ ]:
# Basic summary of weather dataset using describe function from Pandas for the columns that are type float or integer.
weather_pd.describe()

In [ ]:
# Check on the percentage of missing values in each column. We will need to do some data wrangling (e.g., deal with data/time columns) 
# and cleaning (e.g., deal with missing values).
# Use isnull() and sum() function to count missing values in all the columns.
print("Percentage of missing entries for each column:\n", 100.0*(weather_pd.isnull().sum()/len(weather_pd)))

In [ ]:
# Check on the percentage of zero values in each column. Some columns may only contain 0 values so will need to be
# removed.
print("Percentage of zero entries for each column:\n", 100.0*((weather_pd == 0).sum()/len(weather_pd)))

In [ ]:
# Let's see the names of all of the cities in this weather data.
print(weather_pd['city_name'].unique())

In [ ]:
# There seems to be some extra rows of weather data compared with the power data. Let's look at the tail end of
# the weather and power data to compare the end times.
power_pd.tail()

In [ ]:
weather_pd.tail()

The weather data is split into five cities in Spain (Valencia, Madrid, Bilbao, Barcelona, and Seville). Interestingly, despite the power and weather data having the same start and end time/date, the weather data appears to have some extra rows (power data has 35,064 rows, so the weather data should have 35064 * 5 = 175,320 but instead has 178,396, 3,076 more rows than expected. This will need to be investigated further.

The time column for the weather data is in the same format as the power data, YYYY-MM-DD HH:MM:SS+HH:MM, where +HH:MM is the UTC to Central European Time (CET). 

There are also a couple confusing columns in the weather data. These include *temp_min* and *temp_max*, which seem a bit odd given that there is a *temp* column. My guess is the *temp_min* and *temp_max* are the minimum and maximum temperatures, respectively, recorded within each 1 hour time interval and *temp* is the mean temperature at each hour.

The *weather_icon* I believe can be removed as it won't be an informative feature (it's a weather icon code for the TSO website). Also, we will need to convert the categorical weather columns (*weather_main* and *weather_description*) into numerical data via either one-hot encoding or ordinal encoding, depending on if we want an order to weather description from clear to cloudy/rainy. It's also not clear whether we should keep both *weather_main* and *weather_description* or just one of these columns. There is a *weather_id* which also describes the weather using numerical codes, so maybe we won't have to deal with *weather_main* and *weather_description*.

### Bokeh Visualisation Plotting Function

Below I have created a Bokeh plotting function to visualise the power and weather data.

In [ ]:
# Let's visualize power data by plotting power generation/demand for the various power sources along with
# some optional weather features.

# Set up plotting figure. Set x-axis to "datetime" so that the date time can be displayed appropriately.
def explore_plots(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black', color_features='black',
                  labels=None, features_labels=None, symbols=None, symbols_features=None, replaced_columns=None):

    """
    Function to create interactive exploratory plots.
    
    Parameters
    ----------
    dataframe : Pandas dataframe object
        The Pandas dataframe holding the data you want to plot.
    x_axis_column : string
        The name of the column you want to plot along the x-axis.
    y_axis_columns : list
        The names of the columns for the primary data you want to plot along the y-axis.
    x_axis_label : string
        The label to use for the x axis.
    y_axis_label : string
        The label to use for the y axis for the primary data. Note, all column data plotted will have the same y-axis scale
        and label name.
    title : string
        The title for the plot.

    Optional
    --------
    feature_columns : list or string
        The names of the columns you want to plot as additional features along secondary y-axis
        Default: None
    features_ylabel : list or string
        The secondary y-axis feature label/s. If a list greater than one item, list is concatenated into a single string.
        Default: None    
    p : bokeh plotting object
        A bokeh plotting object to add additional plotting objects to.
        Default: None
    normalize : Boolean
        Whether to normalize the data. Caution, might not work well for some features or when including more than one feature.
        Default: False
    other_colors : Boolean
        Whether to use your own colors (True) or to use color names as defined in this function (False).
        Default: False
    color_plot : string or list
        A string or list of colors to use for plotting the primary data. If list, must be length of number y_axis_columns.
        Default: 'black'
    color_features : string or list
        A string or list of colors to use for plotting the features data. If list, must be length of number feature_columns.
        Default: 'black'
    labels : list or string
        The labels to assign all the primary data from the y_axis_columns, should be list of length y_axis_columns if more than one
        primary data is being plotted.
        Default: None
    features_labels : list or string
        The labels to assign all the features data from feature_columns, should be list of length feature_columns if more than one
        feature is being plotted.
        Default: None
    symbols : string or list
        The plotting markers for the primary data. If list, must be length of number of y_axis_columns.
        Default: None
    symbols_features : string or list
        The plotting markers for the features data. If list, must be length of number of feature_columns.
        Default: None
    replaced_columns : Boolean
        Whether to plot the replaced primary data produced through cleaning. The appropriate columns must exist in the dataframe
        if True and given as 'replaced_' and the y_axis_columns, e.g., 'replaced_energy_usage_Wh' for replacements for 
        the 'energy_usage_Wh'.
        Default: False

    Returns
    -------
    p : bokeh object
        The Bokeh plotting object.
    """
    
    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}

    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        # Rotate labels for better readability
        p.xaxis.major_label_orientation = 120
        # Reduce the number of x-axis ticks to avoid crowding
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    # print('labels: ', labels)
    # print('markers: ', symbols)
    # print('colors: ', color_plot_values)
    
    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        # Create a loop to plot the column data for each city in Spain
        for y, city in enumerate(dataframe['city_name'].unique()):

            if normalize:
                max_primary_data = dataframe.loc[dataframe['city_name'] == city, column].max()
            else:
                max_primary_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 | y == 0 else False
            
            # Scatter plot with symbols.
            if symbols:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data, size=10,
                          marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                          legend_label=label + ' ' + str(city), visible=visible)
                
            # Add a line to connect the symbols
            p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                   dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data,
                   line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label + ' ' + str(city),
                   visible=visible)

            if replaced_columns:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, f'replaced_{column}']/max_primary_data,
                          size=10, color="red", marker="diamond", alpha=0.5, legend_label="Replaced " + label + ' ' + str(city),
                          visible=visible)

            # print('marker_index: ', marker_index)
            # print('color_index: ', color_index)
            # print('label_index: ', label_index)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
        label_index = label_index + 1
        

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        if normalize:
            max_features_data = dataframe[feature_columns].max().max()
        else:
            max_features_data = 1

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            # Create a loop to plot the features for each city.
            for y, city in enumerate(dataframe['city_name'].unique()):

                if normalize:
                    max_features_data = dataframe.loc[dataframe['city_name'] == city, feature_column].max()
                else:
                    max_features_data = 1

                # Only show the first column by default, hide others
                visible = True if i == 0 | y == 0 else False
                
                # Scatter plot with symbols. Only plot the energy usage as a function of time for a specific city.
                if symbols_features:
                    p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                              dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data,
                              y_range_name="features", size=10, marker=symbols_features[marker_index],
                              color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label + ' ' + str(city),
                              visible=visible)
                # Add a line to connect the symbols
                p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                       dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data, y_range_name="features",
                       line_width=2, color=color_plot_values[color_index], alpha=0.5,
                       legend_label=feature_label + ' ' + str(city), visible=visible)
            
                marker_index = marker_index + 1
                color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

Before we can proceed with visualising the power and weather data, we need to make a few modifications. The first of which is to transform the time columns in these two datasets to proper datetime format. Additionally, I find it useful to have both utc and local time and by having both, we can remove the +HH:MM attachment on the date/times. Lastly, we will need to create a city column in the power dataframe as a place holder for now as the plotting function expects this (this will be used in overplotting the weather features).

In [ ]:
# Let's make copies of power_pd and weather_pd dataframes.
power_pd1 = power_pd.copy(deep=True)
weather_pd1 = weather_pd.copy(deep=True)

# Rename time columns for consistency in the power and weather dataframes.
power_pd1.rename(columns={'time':'local_time'}, inplace=True)
weather_pd1.rename(columns={'dt_iso':'local_time'}, inplace=True)

# Now convert the date/time strings in the local_time and utc_time columns to datetime64 data type so that I can work with
# the time series data.
power_pd1['local_time_tz'] = pd.to_datetime(power_pd1['local_time'], utc=True)
power_pd1['utc_time'] = power_pd1['local_time_tz'].dt.tz_localize(None)
power_pd1['local_time'] = power_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
power_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = power_pd1.pop('utc_time')
power_pd1.insert(1, 'utc_time', column_utc_time)

weather_pd1['local_time_tz'] = pd.to_datetime(weather_pd1['local_time'], utc=True)
weather_pd1['utc_time'] = weather_pd1['local_time_tz'].dt.tz_localize(None)
weather_pd1['local_time'] = weather_pd1['local_time_tz'].dt.tz_convert('Europe/Madrid').dt.tz_localize(None)
weather_pd1.drop(columns='local_time_tz', inplace=True)
column_utc_time = weather_pd1.pop('utc_time')
weather_pd1.insert(1, 'utc_time', column_utc_time)

In [ ]:
print('Columns and data types in the power dataset: \n', power_pd1.dtypes)
power_pd1.head()

In [ ]:
print('Columns and data types in the weather dataset: \n', weather_pd1.dtypes)
weather_pd1.head()

In [ ]:
# A column for city names in the power dataset and set to 'Spain' for now.
power_pd1['city_name'] = 'Spain'
column_city_name = power_pd1.pop('city_name')
power_pd1.insert(2, 'city_name', column_city_name)

In [ ]:
# Let's create a basic plot exploritory plot showing the energy production as a function of time for one of the
# energy production columns

p = explore_plots(power_pd1, 'local_time', ['generation biomass'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ Biomass\ (MWh)}$$",
                  'Power Generation from Biomass vs Local Time', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['dorange', 'dorange', 'dorange', 'green', 'green', 'green'],
                  color_features=None,
                  labels=['Power Generation from Biomass'], features_labels=None,
                  symbols=['star', 'triangle', 'diamond', 'star', 'triangle', 'diamond'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Power_generation_biomass.html'

title = 'Power Generation from Biomass vs Local Time'

save(p, filename_out, title=title)

Before producing additional plots, let's remove some columns that contain only null or zero values from the power dataset. Once this is done, then we will produce a plot that shows all the power generation columns.

In [ ]:
power_pd2 = power_pd1.copy(deep=True)

# Identify generation columns where all values are NaN and/or 0
null_zero = []
for col in power_pd2.columns:
    if col.startswith('generation'):
        # Create boolean mask where values are NaN or 0
        is_null_or_zero = power_pd2[col].isna() | (power_pd2[col] == 0.0)
        if is_null_or_zero.all():
            null_zero.append(col)

# Print and drop those columns
print("Columns with all null/zero values:")
for col in null_zero:
    print(col)

# Drop from power_pd2
power_pd2.drop(columns=null_zero, inplace=True)

In [ ]:
power_pd2.head()

In [ ]:
# Let's also remove 'forecast wind offshore eday ahead' as all of the values are null.
power_pd2.drop(columns=['forecast wind offshore eday ahead'], inplace=True)
power_pd2.head()

In [ ]:
# Now create a plot showing all the power generation sources as a function of time.
from bokeh.palettes import Category20

# Use Category20 palette (20 distinct colors), pick 14
colors = Category20[20][:14]

# Select 14 distinct Bokeh marker types
available_markers = [
    'circle', 'square', 'triangle', 'diamond', 'inverted_triangle',
    'cross', 'x', 'asterisk', 'circle_cross', 'square_cross',
    'diamond_cross', 'circle_x', 'square_x', 'triangle_dot'
]

# Create a list of all of the power generation columns, labels, colors, and markers.
generation_cols = []
color_plot = []
labels = []
symbols = []
for i, col in enumerate(power_pd2.columns):
    if col.startswith('generation'):
        generation_cols.append(col)
        labels.append(col)  # Just use the column name as label
        color_plot.append(colors[i % len(colors)])  # Safeguard in case >14
        symbols.append(available_markers[i % len(available_markers)])

print("Power generation columns:")
for col in generation_cols:
    print(col)

In [ ]:
p = explore_plots(power_pd2, 'local_time', generation_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MW)}$$",
                  'Power Generation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Power_generation_all_sources.html'

title = 'Power Generation vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(power_pd2, 'local_time', generation_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Normalized\ Power\ Generation}$$",
                  'Normalized Power Generation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=True, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Normalized_Power_generation_all_sources.html'

title = 'Normalized Power Generation vs Local Time in Spain'

save(p, filename_out, title=title)

Many of these power generation sources are cyclical in nature and are likely influenced by weather conditions and demand. There also appears to be some sudden dips where the value is 0 that is common across most of the power sources, so these will need to be investigated further (could be an indication of invalid values).

Next I will create plots showing the total actual load and total actual price. Then I will create plots showing the forecast power generation from solar and wind, forecast of the power load, and forecast for the price which will serve as benchmarks for my ML predictive models.

In [ ]:
p = explore_plots(power_pd2, 'local_time', ['total load actual'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Total\ Power\ Load\ (MW)}$$",
                  'Total Power Demand vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['red'],
                  color_features=None,
                  labels=['total load actual'], features_labels=None,
                  symbols=['circle'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Total_power_load.html'

title = 'Total Power Load vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(power_pd2, 'local_time', ['price actual'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Electricity\ Price\ (EUR/MWh)}$$",
                  'Electricity Price vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['red'],
                  color_features=None,
                  labels=['price actual'], features_labels=None,
                  symbols=['circle'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Electricity_price.html'

title = 'Electricity Price vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(power_pd2, 'local_time', ['forecast solar day ahead', 'forecast wind onshore day ahead'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Forecast\ Power\ Generation\ (MW)}$$",
                  'Forecast Power Generation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['red', 'blue'],
                  color_features=None,
                  labels=['forecast solar day ahead', 'forecast wind onshore day ahead'], features_labels=None,
                  symbols=['circle', 'square'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Forecast_power_generation.html'

title = 'Forecast Power Generation vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(power_pd2, 'local_time', ['total load forecast'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Forecast\ Power\ Load\ (MW)}$$",
                  'Forecast Power Demand vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['green'],
                  color_features=None,
                  labels=['total load forecast'], features_labels=None,
                  symbols=['square'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Forecast_power_load.html'

title = 'Forecast Power Load vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(power_pd2, 'local_time', ['price day ahead'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Forecast\ Electricity\ Price\ (EUR/MWh)}$$",
                  'Forecast Electricity Price vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['green'],
                  color_features=None,
                  labels=['price day ahead'], features_labels=None,
                  symbols=['square'],
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Forecast_electricity_price.html'

title = 'Forecast Electricity Price vs Local Time in Spain'

save(p, filename_out, title=title)

Now for some plots of the weather features.

In [ ]:
# Now create a plot showing numerical weather features as a function of time and city.
from bokeh.palettes import Category20

weather_feats = ['temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg', 'rain_1h', 'rain_3h',
                 'snow_3h', 'clouds_all', 'weather_id']

# Use Category20 palette (20 distinct colors), pick 12
colors = Category20[20][:12]

# Select 12 distinct Bokeh marker types
available_markers = ['circle', 'square', 'triangle', 'diamond', 'inverted_triangle',
    'cross', 'x', 'asterisk', 'circle_cross', 'square_cross',
    'diamond_cross', 'circle_x', 'square_x', 'triangle_dot', 'hex']

# Create a list of all of the colors and markers.
color_plot = []
symbols = []
for i, col in enumerate(weather_feats):
    color_plot.append(colors[i % len(colors)])  # Safeguard in case >14
    symbols.append(available_markers[i % len(available_markers)])

In [ ]:
p = explore_plots(weather_pd1, 'local_time', ['temp', 'temp_min', 'temp_max'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Temperature\ (K)}$$",
                  'Temperature vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False,
                  color_plot=['black', 'black', 'black', 'black', 'black', 'blue', 'blue', 'blue', 'blue', 'blue',
                             'red', 'red', 'red', 'red', 'red'],
                  color_features=None,
                  labels=['temp', 'temp_min', 'temp_max'], features_labels=None,
                  symbols=available_markers,
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Weather_feature_temperature.html'

title = 'Temperature vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(weather_pd1, 'local_time', ['wind_speed'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Wind\ Speed\ (m\,s^{-1})}$$",
                  'Wind Speed & Direction vs Local Time in Spain', feature_columns=['wind_deg'],
                  features_ylabel='Wind Direction (deg)', 
                  p=None, normalize=False, other_colors=False,
                  color_plot=['black', 'blue', 'orange', 'red', 'green'],
                  color_features=['violet', 'pink', 'cyan', 'dorange', 'dgrey'],
                  labels=['wind_speed'], features_labels=['wind_deg'],
                  symbols=available_markers,
                  symbols_features=available_markers)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Weather_feature_wind.html'

title = 'Wind Speed & Direction vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(weather_pd1, 'local_time', ['rain_1h', 'rain_3h', 'snow_3h'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Precipitation\ (mm)}$$",
                  'Precipitation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, 
                  p=None, normalize=False, other_colors=False,
                  color_plot=['green', 'green', 'green', 'green', 'green', 'orange', 'orange', 'orange', 'orange', 'orange',
                             'blue', 'blue', 'blue', 'blue', 'blue'],
                  color_features=None,
                  labels=['rain_1h', 'rain_3h', 'snow_3h'], features_labels=None,
                  symbols=available_markers,
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Weather_feature_precipitation.html'

title = 'Precipitation & Direction vs Local Time in Spain'

save(p, filename_out, title=title)

In [ ]:
p = explore_plots(weather_pd1, 'local_time', ['clouds_all'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Clouds\ (\%)}$$",
                  'Clouds vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, 
                  p=None, normalize=False, other_colors=False,
                  color_plot=['black', 'blue', 'orange', 'red', 'green'],
                  color_features=None,
                  labels=['clouds_all'], features_labels=None,
                  symbols=available_markers,
                  symbols_features=None)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Weather_feature_clouds.html'

title = 'Clouds vs Local Time in Spain'

save(p, filename_out, title=title)

### Step Four: Data Cleaning

Now it is time to perform some data cleaning involving the removal and replacement of null values as well as the identification, removal, and replacement of outlier values in both the power and weather datasets. Outlier values will be the most difficult to deal with. Additionally, I will also want to check that the time columns in both the power and weather datasets are aligned so that the two datasets can be merged. one complication is that the weather data is provided for five cities in Spain whereas the power data is for all of Spain.

First I'll check the number of time rows for each city in the weather dataset and compare that to the number of time rows in the power dataset.

In [ ]:
print('Number of local time rows in the power dataset: ', len(power_pd2['local_time']))
print('Number of unique local time rows in the power dataset: ', len(power_pd2['local_time'].unique()))

print('Number of utc time rows in the power dataset: ', len(power_pd2['utc_time']))
print('Number of unique utc time rows in the power dataset: ', len(power_pd2['utc_time'].unique()))

print(' ')

for city in weather_pd1['city_name'].unique():
    print('Number of utc time rows in the weather dataset: ', len(weather_pd1.loc[weather_pd1['city_name'] == city, 'utc_time']),
         ' in city: ', city)
    print('Number of unique utc time rows in the weather dataset: ',
          len(weather_pd1.loc[weather_pd1['city_name'] == city, 'utc_time'].unique()),
         ' in city: ', city)
    print(' ')

For some reason there are duplicated time rows for the weather data so I will explore this further.

In [ ]:
cities = weather_pd1['city_name'].unique()

for city in cities:
    city_df = weather_pd1[weather_pd1['city_name'] == city]
    duplicated_times = city_df[city_df.duplicated(subset='utc_time', keep=False)]
    
    if not duplicated_times.empty:
        print(f"\nDuplicate utc_time entries in city: {city}")
        print(duplicated_times.sort_values(by='utc_time').head(10))  # show first 10 for brevity

So it appears that there are duplicate rows because when two different weather conditions are reported within the same hour, the time stamp is duplicated. We will need to resolve this duplication issue before we can merge the weather and power datasets. I will explore the duplications further here. Since it appears the *weather_id* column can be used to describe the weather conditions, I'm thinking we can drop the categorical weather columns *weather_main*, *weather_description*, and *weather_icon*. Before I do that, I want to see what all of the unique *weather_id* values correspond to *weather_main* and *weather_description*.

In [ ]:
weather_ids = weather_pd1['weather_id'].unique()

for weather_id in weather_ids:
    weather_main = weather_pd1.loc[weather_pd1['weather_id'] == weather_id, 'weather_main'].unique()
    weather_description = weather_pd1.loc[weather_pd1['weather_id'] == weather_id, 'weather_description'].unique()
    print('Weather ID: ', weather_id)
    print('Weather Main: ', weather_main)
    print('Weather Description: ', weather_description)
    print(' ')

I think the best way to proceed is to select the most extreme weather condition in each duplicate time stamp (which seems to come in pairs) by defining a weather severity mapping (the weather mapping condition codes come from [OpenWeatherMap](https://openweathermap.org/weather-conditions)) and selecting the most extreme weather condition from it and removing the other duplicate time stamps. This method isn't perfect but I think is reasonable given the relatively small number of duplicate time stamps.

In [ ]:
# Weather severity mapping that is used to help select the most severe weather conditions from two duplicate
# time stamps.

def get_weather_severity(weather_id):
    if 200 <= weather_id < 300:
        return 5  # Thunderstorm
    elif 300 <= weather_id < 400:
        return 3  # Drizzle
    elif 500 <= weather_id < 600:
        return 4  # Rain
    elif 600 <= weather_id < 700:
        return 4  # Snow
    elif 700 <= weather_id < 800:
        return 2  # Atmosphere (mist, fog)
    elif weather_id == 800:
        return 0  # Clear
    elif 801 <= weather_id <= 804:
        return 1  # Clouds
    else:
        return -1  # Unknown or invalid


In [ ]:
weather_pd2 = weather_pd1.copy(deep=True)

weather_pd2['severity'] = weather_pd2['weather_id'].apply(get_weather_severity)

# Sort by severity and weather_id
weather_pd2 = weather_pd2.sort_values(by=['city_name', 'utc_time', 'severity', 'weather_id'], ascending=[True, True, False, False])

# Identify the index of the rows that will be kept
keep_index = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').index

# Create the audit dataframe, mark rows as kept or dropped
weather_duplicates_audit = weather_pd2.copy(deep=True)
weather_duplicates_audit['keep_row'] = weather_duplicates_audit.index.isin(keep_index)

# Drop duplicates keeping most severe (and then highest ID) row
weather_pd2 = weather_pd2.drop_duplicates(subset=['city_name', 'utc_time'], keep='first').reset_index(drop=True)
weather_duplicates_audit.reset_index(drop=True, inplace=True)

In [ ]:
cities = weather_duplicates_audit['city_name'].unique()

for city in cities:
    city_df = weather_duplicates_audit[weather_duplicates_audit['city_name'] == city]
    duplicated_times = city_df[city_df.duplicated(subset='utc_time', keep=False)]
    
    if not duplicated_times.empty:
        print(f"\nDuplicate utc_time entries in city: {city}")
        print(duplicated_times.sort_values(by='utc_time').head(10))  # show first 10 for brevity

In [ ]:
for city in weather_pd2['city_name'].unique():
    print('Number of utc time rows in the weather dataset: ', len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time']),
         ' in city: ', city)
    print('Number of unique utc time rows in the weather dataset: ',
          len(weather_pd2.loc[weather_pd2['city_name'] == city, 'utc_time'].unique()),
         ' in city: ', city)
    print(' ')

I've removed the duplicate time stamps for each city.

Next steps, replace missing values in the power and weather datasets.

#### Missing values in weather data
Now I'll check if there are any missing (null) values in the weather data.

In [ ]:
print("Number of missing entries for each column:\n", weather_pd2.isnull().sum())

#### Missing values in Power data
Now I'll check if there are any missing (null) values in the power data.

In [ ]:
print("Percentage of missing entries for each column:\n", 100.0*(power_pd2.isnull().sum()/len(power_pd2)))

In [ ]:
print("Percentage of zero entries for each column:\n", 100.0*((power_pd2 == 0).sum()/len(power_pd2)))

#### Imputation of Power Dataset
There are no missing values in the weather data which makes our task easier. However, the power dataset has missing values. Let's take care of these using a hybrid approach as a combination of a rolling averaging window for small data gaps (3 or less missing values) and a local regression method (LOESS, locally estimated scatterplot smoothing) to fit smooth curves through the data replace larger gaps. This method works by leveraging the information from nearby data points to utilize data from similar observations to fill in the gaps where data is missing. I found this method worked reasonably well with another power dataset I worked with.

The above approach generally works well with the exception of solar power. To handle solar power generation (which is very cyclical in nature), missing values during the daytime (assumed to be between 6 and 18 hours for simplicity) will be replaced by fitting a sine curve through three days of data (day before and after) if there are more than two consecutive data points missing. If there are only one or two consecutive data points missing during the daytime, a simple linear interpolation is used instead. For missing data during the nighttime, the median solar power generation is calculated and used as a replacement (this is done because nighttime solar power does not drop to zero but there is some residual power output (probably from batteries).

In [ ]:
from scipy.optimize import curve_fit

# Sine-like daily solar generation curve
def daily_solar_curve(hour, amplitude, phase_shift, vertical_shift):
    return amplitude * np.sin((np.pi / 12) * (hour - phase_shift)) + vertical_shift


def get_median_night_residual(df, date_col, hour_col, solar_col, target_date):
    """
    Returns the median nighttime value for the closest night to target_date in the dataframe.
    Nighttime is defined as hours < 6 or >= 18.
    """
    df_night = df[
        ((df[hour_col] < 6) | (df[hour_col] >= 18)) &
        (df[date_col].isin([target_date - pd.Timedelta(days=1), target_date, target_date + pd.Timedelta(days=1)]))
    ]
    # Select night closest to target_date
    night_dates = df_night[date_col].unique()
    if len(night_dates) == 0:
        return 0  # Fallback
    closest_night = min(night_dates, key=lambda d: abs((d - target_date).days))
    return df_night.loc[df_night[date_col] == closest_night, solar_col].median()

def fill_solar_with_rolling_fit(df, time_col, solar_col, filled_col='filled_generation solar',
                                replaced_col='replaced_generation_solar'):
    """
    Fills missing values in the solar generation column using a rolling 3-day window sinusoidal fit.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataframe containing the power generation data.
    time_col : str
        Name of the datetime column (must be datetime64).
    solar_col : str
        Name of the solar generation column.
    filled_col : str
        Name of the column where the filled (original + replacement) data is stored.
    replaced_col : str
        Name of the column where only the replacement values are stored.

    Returns
    ----------
    filled_data : pd.Series
        The column with the filled (original + replacement) data.
    replaced_data : pd.Series
        A column with only the replaced values (NaN elsewhere).
    """
    df = df.copy()
    df['hour'] = df[time_col].dt.hour
    df['date'] = df[time_col].dt.date
    filled_data = df[filled_col].copy()           # Column with the filled data
    replaced_data = df[replaced_col].copy()       # Column with the replaced data

    unique_dates = np.array(sorted(df['date'].unique()))
    is_daytime = lambda h: (h >= 6) & (h <= 18)

    # Iterate through center days for a 3-day rolling window
    for i in range(1, len(unique_dates) - 1):
        window_dates = unique_dates[i-1:i+2]
        center_day = unique_dates[i]

        mask_window = df['date'].isin(window_dates)
        window_df = df[mask_window & df[filled_col].notna()]
        window_df = window_df[is_daytime(window_df['hour'])]
        #window_df = window_df[(window_df['hour'] >= 6) & (window_df['hour'] <= 20)]

        if len(window_df) < 18:
            continue

        try:
            popt, _ = curve_fit(
                daily_solar_curve,
                window_df['hour'],
                window_df[filled_col],
                p0=[window_df[filled_col].max(), 12, 0],
                maxfev=10000
            )
        except Exception:
            continue

        mask_center = (
            (df['date'] == center_day) &
            (df[filled_col].isna()) &
            is_daytime(df['hour'])
        )
        
        # mask_center_day = (
        #     (df['date'] == center_day) &
        #     (df[filled_col].isna()) &
        #     (df['hour'] >= 6) & (df['hour'] <= 20)
        # )

        # if mask_center_day.any():
        #     fitted_vals = daily_solar_curve(df.loc[mask_center_day, 'hour'], *popt)
        #     filled_data.loc[mask_center_day] = fitted_vals
        #     replaced_data.loc[mask_center_day] = fitted_vals

        # Identify consecutive NaN gaps (within the center day daytime)
        subidx = df[mask_center].index
        # We'll group gaps by consecutive indices (daytime NaNs)
        if not subidx.empty:
            groups = np.split(subidx, np.where(np.diff(subidx) != 1)[0]+1)
            for group in groups:
                if len(group) <= 2:
                    # Interpolate for short gaps
                    interp_vals = filled_data.interpolate(method='linear').loc[group]
                    filled_data.loc[group] = interp_vals
                    replaced_data.loc[group] = interp_vals
                else:
                    # Use curve fit for longer gaps
                    hours = df.loc[group, 'hour']
                    fitted_vals = daily_solar_curve(hours, *popt)
                    max_val = window_df[filled_col].max()
                    # Use median night value for negative fits
                    for idx, val, hr in zip(group, fitted_vals, hours):
                        if val < 0:
                            median_night_val = get_median_night_residual(df, 'date', 'hour', filled_col, center_day)
                            filled_data.loc[idx] = median_night_val
                            replaced_data.loc[idx] = median_night_val
                        else:
                            clipped_val = min(val, max_val)
                            filled_data.loc[idx] = clipped_val
                            replaced_data.loc[idx] = clipped_val

    return filled_data, replaced_data

In [ ]:
# Using the hybrid approach with local regression (LOESS) to fill in large gaps.
# I will use loess_1d function from loess as part of the Pypi library (https://pypi.org/project/loess/).
from loess.loess_1d import loess_1d

# Rolling average window size.
rolling_window = 5

# Make a deep copy of the dataframe in case of accidental modifications
power_pd_clean = power_pd2.copy(deep=True)

columns_to_clean = [
    'generation biomass',
    'generation fossil brown coal/lignite',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation fossil oil',
    'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage',
    'generation hydro water reservoir',
    'generation nuclear',
    'generation other',
    'generation other renewable',
    'generation solar',
    'generation waste',
    'generation wind onshore',
    'forecast solar day ahead',
    'forecast wind onshore day ahead',
    'total load forecast',
    'total load actual',
    'price day ahead',
    'price actual'
]

# Ensure datetime column is available as integer timestamps for LOESS
power_pd_clean['utc_timestamp'] = power_pd_clean['utc_time'].astype('int64') // 10**9

# Replace negative values with NaN (assumed invalid)
for col in columns_to_clean:
    power_pd_clean.loc[power_pd_clean[col] < 0, col] = np.nan

# Apply hybrid imputation to each column
for col in columns_to_clean:
    print(f"Processing column: {col}")
    
    # Create new columns to track replacement and filled values
    power_pd_clean[f'replaced_{col}'] = np.where(power_pd_clean[col].isna(), True, np.nan)
    power_pd_clean[f'filled_{col}'] = power_pd_clean[col].copy()

    #Need to handle solar power generation separately to the other columns as the LOESS approach does not work well.
    if col == 'generation solar':
        # Use the fill_solar_with_rolling_fit function to fill the missing values for the solar generation column.
        solar_filled, solar_replaced = fill_solar_with_rolling_fit(power_pd_clean, 'local_time', 'generation solar',
                                                                   filled_col='filled_generation solar',
                                                                   replaced_col='replaced_generation solar')
        power_pd_clean[f'filled_{col}'] = solar_filled
        power_pd_clean[f'replaced_{col}'] = solar_replaced
        
    else:
        # Step 1: Rolling average for small gaps
        rolling_filled = power_pd_clean[col].rolling(window=rolling_window, center=True, min_periods=1).mean()
        small_gap_mask = power_pd_clean[col].isna()
        power_pd_clean.loc[small_gap_mask, f'filled_{col}'] = rolling_filled[small_gap_mask]
        power_pd_clean.loc[small_gap_mask, f'replaced_{col}'] = rolling_filled[small_gap_mask]
      
        # Step 2: LOESS for medium gaps
        subset = power_pd_clean.copy()
        subset['utc_timestamp'] = power_pd_clean['utc_timestamp']
        mask_loess = subset[f'filled_{col}'].isna()
    
        valid_idx = subset[col].notna()
        missing_idx = subset[col].isna()
        loess_frac = 0.0005
    
        if valid_idx.sum() > 10 and missing_idx.sum() > 0:
            valid_timestamps = subset.loc[valid_idx, 'utc_timestamp'].to_numpy()
            valid_values = subset.loc[valid_idx, col].to_numpy()
            missing_timestamps = subset.loc[missing_idx, 'utc_timestamp'].to_numpy()
    
            #print(missing_timestamps)
    
            try:
                time_out, loess_fitted, _ = loess_1d(valid_timestamps, valid_values, xnew=missing_timestamps, frac=loess_frac)
    
                if len(loess_fitted) > 0:
                    loess_df = pd.DataFrame({'utc_timestamp': missing_timestamps, 'loess_value': loess_fitted})
                    loess_df.set_index(subset.loc[missing_idx].index, inplace=True)
        
                    power_pd_clean.loc[mask_loess, f'filled_{col}'] = loess_df['loess_value']
                    power_pd_clean.loc[mask_loess, f'replaced_{col}'] = loess_df['loess_value']
    
            except Exception as e:
                print(f"⚠️ LOESS failed for column '{col}': {e}")
                # Fallback to linear interpolation
                interpolated_values = power_pd_clean[col].interpolate(method='linear')
                power_pd_clean.loc[mask_loess, f'filled_{col}'] = interpolated_values[mask_loess]
                power_pd_clean.loc[mask_loess, f'replaced_{col}'] = interpolated_values[mask_loess]

    # Step 3: Final fallback — linear interpolation
    final_mask = power_pd_clean[f'filled_{col}'].isna()
    power_pd_clean.loc[final_mask, f'filled_{col}'] = power_pd_clean[col].interpolate(method='linear')[final_mask]
    power_pd_clean.loc[final_mask, f'replaced_{col}'] = power_pd_clean[f'filled_{col}'][final_mask]

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_dataset_missing_LOESS_replaced.csv'
power_pd_clean.to_csv(filename_out, index=False)

In [ ]:
# Now create a plot indicating the replacements.

colors = Category20[20][:20]

# Select 20 distinct Bokeh marker types
available_markers = [
    'circle', 'square', 'triangle', 'diamond', 'inverted_triangle',
    'cross', 'x', 'asterisk', 'circle_cross', 'square_cross',
    'diamond_cross', 'circle_x', 'square_x', 'triangle_dot',
    'plus', 'hex', 'square_dot', 'diamond_dot', 'star', 'star_dot'
]

gen_cols = [
    'generation biomass',
    'generation fossil brown coal/lignite',
    'generation fossil gas',
    'generation fossil hard coal',
    'generation fossil oil',
    'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage',
    'generation hydro water reservoir',
    'generation nuclear',
    'generation other',
    'generation other renewable',
    'generation solar',
    'generation waste',
    'generation wind onshore',
]

forecast_gen_cols = [
    'forecast solar day ahead',
    'forecast wind onshore day ahead'
]

load_cols = [
    'total load forecast',
    'total load actual'
]

price_cols = [
    'price day ahead',
    'price actual'
]

# Create a list of all of the columns, labels, colors, and markers.
plot_cols = []
color_plot = []
labels = []
symbols = []
for i, col in enumerate(gen_cols):
    plot_cols.append(col)
    labels.append(col)  # Just use the column name as label
    color_plot.append(colors[i % len(colors)])  # Safeguard in case >20
    symbols.append(available_markers[i % len(available_markers)])

p = explore_plots(power_pd_clean, 'local_time', plot_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MW)}$$",
                  'Power Generation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None,
                  replaced_columns=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Power_gen_LOESS_replaced.html'

title = 'Power Generation vs Local Time with the LOESS Replacements'

save(p, filename_out, title=title)

In [ ]:
# Create a list of all of the columns, labels, colors, and markers.
plot_cols = []
color_plot = []
labels = []
symbols = []
for i, col in enumerate(forecast_gen_cols):
    plot_cols.append(col)
    labels.append(col)  # Just use the column name as label
    color_plot.append(colors[i % len(colors)])  # Safeguard in case >20
    symbols.append(available_markers[i % len(available_markers)])

p = explore_plots(power_pd_clean, 'local_time', plot_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Forecast\ Power\ Generation\ (MW)}$$",
                  'Forecast Power Generation vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None,
                  replaced_columns=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Forecast_power_gen_LOESS_replaced.html'

title = 'Forecast Power Generation vs Local Time with the LOESS Replacements'

save(p, filename_out, title=title)

In [ ]:
# Create a list of all of the columns, labels, colors, and markers.
plot_cols = []
color_plot = []
labels = []
symbols = []
for i, col in enumerate(load_cols):
    plot_cols.append(col)
    labels.append(col)  # Just use the column name as label
    color_plot.append(colors[i % len(colors)])  # Safeguard in case >20
    symbols.append(available_markers[i % len(available_markers)])

p = explore_plots(power_pd_clean, 'local_time', plot_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Load\ (MW)}$$",
                  'Power Load vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None,
                  replaced_columns=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Power_load_LOESS_replaced.html'

title = 'Power Load vs Local Time with the LOESS Replacements'

save(p, filename_out, title=title)

In [ ]:
# Create a list of all of the columns, labels, colors, and markers.
plot_cols = []
color_plot = []
labels = []
symbols = []
for i, col in enumerate(price_cols):
    plot_cols.append(col)
    labels.append(col)  # Just use the column name as label
    color_plot.append(colors[i % len(colors)])  # Safeguard in case >20
    symbols.append(available_markers[i % len(available_markers)])

p = explore_plots(power_pd_clean, 'local_time', plot_cols,
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Energy\ Price\ (EUR/MWh)}$$",
                  'Energy Price vs Local Time in Spain', feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=True,
                  color_plot=color_plot,
                  color_features=None,
                  labels=labels, features_labels=None,
                  symbols=symbols,
                  symbols_features=None,
                  replaced_columns=True)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Price_LOESS_replaced.html'

title = 'Energy Price vs Local Time with the LOESS Replacements'

save(p, filename_out, title=title)

This replacement method seems to have worked well.

In [ ]:
power_pd_clean.columns

In [ ]:
columns_final = ['local_time', 'utc_time', 'utc_timestamp', 'city_name',  
       'filled_generation biomass',
       'filled_generation fossil brown coal/lignite',
       'filled_generation fossil gas',
       'filled_generation fossil hard coal',
       'filled_generation fossil oil',
       'filled_generation hydro pumped storage consumption',
       'filled_generation hydro run-of-river and poundage',
       'filled_generation hydro water reservoir',
       'filled_generation nuclear',
       'filled_generation other',
       'filled_generation other renewable',
       'filled_generation solar',
       'filled_generation waste',
       'filled_generation wind onshore',
       'filled_forecast solar day ahead',
       'filled_forecast wind onshore day ahead',
       'filled_total load forecast',
       'filled_total load actual',
       'filled_price day ahead',
       'filled_price actual']

power_pd_final = power_pd_clean[columns_final].copy(deep=True)

filled_cols = [
    'filled_generation biomass',
    'filled_generation fossil brown coal/lignite',
    'filled_generation fossil gas',
    'filled_generation fossil hard coal',
    'filled_generation fossil oil',
    'filled_generation hydro pumped storage consumption',
    'filled_generation hydro run-of-river and poundage',
    'filled_generation hydro water reservoir',
    'filled_generation nuclear',
    'filled_generation other',
    'filled_generation other renewable',
    'filled_generation solar',
    'filled_generation waste',
    'filled_generation wind onshore',
    'filled_forecast solar day ahead',
    'filled_forecast wind onshore day ahead',
    'filled_total load forecast',
    'filled_total load actual',
    'filled_price day ahead',
    'filled_price actual'
]

rename_dict = {col: col.replace('filled_', '') for col in filled_cols}

power_pd_final.rename(columns=rename_dict, inplace=True)

In [ ]:
power_pd_final.columns

In [ ]:
print("Percentage of missing entries for each column:\n", 100.0*(power_pd_final.isnull().sum()/len(power_pd_final)))

No more missing values in any of the columns. Now ready to combine weather and power datasets!

### Step 5: Convert categorical data to numerical data
While the weather contains some categorical columns, those columns have associated numerical columns (e.g., 'weather_id' is the numerical translation of 'weather_main' and 'weather_description' is a more descriptive version of 'weather_main'. 'weather_icon' is not important as it describes the weather icon used in weather forecast so can be ignored. All other columns are numerical so we can skip this part.

### Step 6: Merge weather and power datasets
The weather dataset has weather conditions across five cities in Spain while the power dataset is all of Spain, so a many-to-one relationship. So to merge the datasets, I will use the merge function in Pandas on utc time ('utc_time') to broadcast the energy data across the five cities for each timestamp. Thankfully the weather and power datasets are on the same timestamps, so no need to interpolate or resample one dataset to match the timestamps of the other.

In [ ]:
merged_power_weather = pd.merge(weather_pd2, power_pd_final, on='utc_time', how='left')

In [ ]:
merged_power_weather.columns

In [ ]:
# Some extra columns were created through the merge process: local_time_x, city_name_x, local_time_y, and city_name_y.
# local_time_x and local_time_y should be the same so I can delete one of them and rename the other as simply 'local_time', 
# but I need to verify local_time_x and local_time_y are indeed the same. city_name_x should the be the cities in Spain from the weather
# dataset whereas city_name_y was a place holder column in the power dataset with the city names set to Spain. We can now remove the
# city_name_y column as it is not needed and rename city_name_x to city_name.

print(merged_power_weather['local_time_x'].equals(merged_power_weather['local_time_y']))

print('Weather data local time: ', merged_power_weather['local_time_x'],
      '. Power data local time: ', merged_power_weather['local_time_y'])

print('Weather data city names: ', merged_power_weather['city_name_x'], 
      '. Power data city names: ', merged_power_weather['city_name_y'])

In [ ]:
merged_power_weather.drop(columns=['local_time_y', 'city_name_y'], axis=1, inplace=True)
merged_power_weather.rename(columns={'local_time_x': 'local_time', 'city_name_x': 'city_name'}, inplace=True)
print(merged_power_weather)

### Step 7: Initial Correlation Analysis
I'll use Pearson and Spearman correlation heatmap matrices to check correlations between the energy generation/demand columns and weather features. Additionally I'll check correlations between pricing, demand, and weather. The correlations will be measured for each city separately to assess how local weather in each city correlates with the national energy generation, load, and pricing.

In [ ]:
# Create Pearson and Spearman correlation heatmap matrices

# Import the spearman correlation function from the Scipy stats library.
from scipy.stats import spearmanr

# Need to specify the specifc power (usage and generation) and weather-related features. Let's look at renewable specifically,
# solar, wind, and other renewable along with load and price.
# power_load_price_features = ['generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
#                              'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
#                              'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation nuclear',
#                              'generation other', 'generation other renewable', 'generation solar', 'generation waste',
#                              'generation wind onshore', 'total load actual', 'price actual']

power_load_price_features = ['generation other renewable', 'generation solar', 'generation wind onshore',
                             'total load actual', 'price actual']

time_weather_other_features = ['temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
                               'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'weather_id', 'severity']

# Make a deep copy of the dataframe in case of accidental modifications
merged_power_weather_copy = merged_power_weather.copy(deep=True)

# The unique cities in the dataset.
cities = merged_power_weather_copy['city_name'].unique()

correlation_results = {}

for city in cities:
    subset = merged_power_weather_copy[merged_power_weather_copy['city_name'] == city]

    # Use only numeric weather/time columns + power columns
    features = power_load_price_features + time_weather_other_features
    
    subset = subset[features].dropna()
    if subset.empty:
        print(f"Skipping city {city} due to insufficient data.")
        continue

    pearson_corr = subset.corr(method='pearson')
    spearman_corr = subset.corr(method='spearman')

    # Compute Spearman p-values
    p_values = np.zeros((len(features), len(features)))
    for i, col1 in enumerate(features):
        for j, col2 in enumerate(features):
            if i != j:
                _, p = spearmanr(subset[col1], subset[col2])
                p_values[i, j] = p

    p_values_df = pd.DataFrame(p_values, index=features, columns=features)
    
    correlation_results[city] = {'pearson': pearson_corr, 'spearman': spearman_corr, 'p_values': p_values_df}

    # Plot heatmaps
    fig, axes = plt.subplots(1, 2, figsize=(24, 14))
    sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
    axes[0].set_title(f'Pearson Correlation - City {city}')
    sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1])
    axes[1].set_title(f'Spearman Correlation - City {city}')

    direct_out = current_dir + '/output/exploratory/correlations/'
    if not os.path.exists(direct_out):
        os.makedirs(direct_out)
    filename_out = os.path.join(direct_out, f'Pearson_Spearman_heatmap_matrices_city_{city}.pdf')
    plt.tight_layout()
    plt.savefig(filename_out)
    plt.show()
    plt.close()

# Store and display Spearman correlation results for further inspection
correlation_data = pd.concat({k: v['spearman'] for k, v in correlation_results.items()}, names=['City'])

In [ ]:
# Do the same things as above but as 1D heatmaps, showing the correlations between the weather features (in the rows) and
# power/load/price as columns.

from scipy.stats import spearmanr

# Define your selected features
power_load_price_features = ['generation other renewable', 'generation solar', 'generation wind onshore',
                             'total load actual', 'price actual']

time_weather_other_features = ['temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
                               'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'weather_id', 'severity']

merged_power_weather_copy = merged_power_weather.copy(deep=True)
cities = merged_power_weather_copy['city_name'].unique()

correlation_results = {}

for city in cities:
    subset = merged_power_weather_copy[merged_power_weather_copy['city_name'] == city]
    subset = subset[power_load_price_features + time_weather_other_features].dropna()

    if subset.empty:
        print(f"Skipping city {city} due to insufficient data.")
        continue

    # 2D matrix: weather rows × power columns
    pearson_corr = pd.DataFrame(
        {pf: [subset[pf].corr(subset[wf], method='pearson') for wf in time_weather_other_features]
         for pf in power_load_price_features},
        index=time_weather_other_features
    )

    spearman_corr = pd.DataFrame(
        {pf: [subset[pf].corr(subset[wf], method='spearman') for wf in time_weather_other_features]
         for pf in power_load_price_features},
        index=time_weather_other_features
    )

    # Compute p-values for Spearman
    spearman_pvals = pd.DataFrame(
        {pf: [spearmanr(subset[pf], subset[wf]).pvalue for wf in time_weather_other_features]
         for pf in power_load_price_features},
        index=time_weather_other_features
    )

    correlation_results[city] = {
        'pearson': pearson_corr,
        'spearman': spearman_corr,
        'spearman_pvals': spearman_pvals
    }

    # Plot Pearson and Spearman heatmaps side by side
    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(time_weather_other_features))))
    sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
    axes[0].set_title(f'Pearson Correlation: City {city}')
    axes[0].set_ylabel('Weather Feature')
    axes[0].set_xlabel('Power/Load/Price Feature')

    # For Spearman: add p-values in parentheses if you want
    spearman_annot = spearman_corr.copy().astype(str)
    for pf in power_load_price_features:
        for wf in time_weather_other_features:
            val = spearman_corr.loc[wf, pf]
            pval = spearman_pvals.loc[wf, pf]
            spearman_annot.loc[wf, pf] = f"{val:.2f}\n(p={pval:.2g})"

    sns.heatmap(spearman_corr, annot=spearman_annot, fmt="", cmap='coolwarm', ax=axes[1])
    axes[1].set_title(f'Spearman Correlation: City {city}')
    axes[1].set_ylabel('Weather Feature')
    axes[1].set_xlabel('Power/Load/Price Feature')

    plt.tight_layout()
    
    direct_out = current_dir + '/output/exploratory/correlations/'
    if not os.path.exists(direct_out):
        os.makedirs(direct_out)
    filename_out = os.path.join(direct_out, f'Pearson_Spearman_1D_heatmap_city_{city}.pdf')
    plt.savefig(filename_out)
    plt.show()
    plt.close()

# Store and display correlation results in a table format
correlation_data2 = pd.concat({k: v['spearman'] for k, v in correlation_results.items()}, names=['City'])

Based on these correlation plots, solar generation has a strong positive correlation with temperature and a strong negative correlation with humidity. This makes sense as hotter days tend to be less cloudy and daylight hours are likely to be longer (i.e., summer) while days that are more humid tend to be cloudier and wetter, so less sunshine. Wind generation tends to be strongly correlated with wind speed, which makes sense though somewhat negatively correlated with temperature (possibly windier on cooler days, probably in winter associated with cold fronts). Power load tends to be higher on warmer days, likely due to increased air conditioning usage.

### Step 8: Feature Engineering
Now I'm going to generate some new features that I feel will strengthen the predictive power of ML models. Here are the features I will engineer:
* Solar flux as a function of time and city. Given that solar energy generation was highly correlated with temperature and temperature should be highly correlated with solar flux, it makes sense to engineer this feature. I suspect that solar generation will be even more correlated with solar flux than temperature. Additinally, solar flux should also correlate strongly with energy demand.
* Sunrise and sunset times as this will be important for demand and power generation.
* Sunrise and sunset as fractions of a day.
* Length of daylight.
* Local time as a fraction of the day so that we can use the time as a model feature.
* Local date as a fraction of a year.
* Day of week using one-hot encoding.
* Workday and weekend using one-hot encoding.
* Convert local time and time of week to cyclical times using sine and cosine for periodicity as this might help the model to better understand the time-based repeating nature of energy usage and generation more easily.

In [ ]:
# Now I'll create a function to compute the solar irradiance as a function of local time and city.
import pvlib
from pvlib.location import Location
import pgeocode

def calculate_solar_flux(row):
    location = pvlib_locs[row['city_name']]
    # Get the position of the Sun, specifically the zenith elevation.
    solpos = location.get_solarposition(row['utc_time'])
    # Get the day of the year which is used for calculating Sun's position.
    day_of_year = row['local_time'].dayofyear
    # Use 1361Wm^2  as the solar radiation 'constant'.
    dni = pvlib.irradiance.disc(1361.0, solpos['zenith'], day_of_year)['dni']
    return dni.item()     # Convert single element numpy array to float when returning

In [ ]:
merged_power_weather_feat = merged_power_weather.copy(deep=True)

# tqdm progress bar for pandas operations.
tqdm.pandas()

# Precompute city coordinate and put in dictionary.
city_names = merged_power_weather_feat['city_name'].unique()
nomi = pgeocode.Nominatim('ES')
city_coords = {}

for city in city_names:
    result = nomi.query_location(city)
    # Take the first row (if Series, convert; if DataFrame, take .iloc[0])
    if hasattr(result, 'iloc'):
        lat = float(result.iloc[0]['latitude'])
        lon = float(result.iloc[0]['longitude'])
    else:  # If already Series
        lat = float(result['latitude'])
        lon = float(result['longitude'])
    city_coords[city] = (lat, lon)

pvlib_locs = {city: Location(lat, lon) for city, (lat, lon) in city_coords.items()}

In [ ]:
# Calculate the solar flux using pvlib and pgeocode by calling the calculate_solar_flux function I created in the previous cell.
# Instead of the normal .apply Pandas operation, I'll using progress_apply as the following takes some time (about 30 minutes).
merged_power_weather_feat['solar_flux_Wm2'] = merged_power_weather_feat.progress_apply(calculate_solar_flux, axis=1)

In [ ]:
# Plot solar generation and solar flux as weather feature
p = explore_plots(merged_power_weather_feat, 'local_time', ['generation solar'],
                  r"$$\mathrm{Local\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ Solar\ (MWh)}$$",
                  'Power Generation from Solar vs Local Time and Solar Flux', feature_columns=['solar_flux_Wm2'],
                  features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"], p=None, normalize=False, other_colors=False,
                  color_plot=['dorange', 'dorange', 'dorange', 'dorange', 'dorange'],
                  color_features=['yellow', 'yellow', 'yellow', 'yellow', 'yellow'],
                  labels=['Power Generation from Solar'], features_labels=['Solar Flux'],
                  symbols=['star', 'triangle', 'diamond', 'square', 'circle'],
                  symbols_features=['star', 'triangle', 'diamond', 'square', 'circle'],
                  replaced_columns=False)

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Solar_power_vs_solar_flux.html'

title = 'Power Generation from Solar vs Local Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
# Let's add the additional time/date features.
import holidays
from suntime import Sun
from datetime import date, datetime, timezone

# Compute the meteorological season based on month
def get_season(month):
    if month in [3, 4, 5]:
        # Spring
        return 1
    elif month in [6, 7, 8]:
        # Summer
        return 2
    elif month in [9, 10, 11]:
        # Autumn
        return 3
    else:
        # Winter
        return 0

# Determine the sunrise and sunset times.
def precompute_sunrise_sunset(city_coords_dict, all_dates):
    records = []
    for city, (lat, lon) in city_coords_dict.items():
        sun = Sun(lat, lon)
        for d in all_dates:
            # Make sure d is a datetime.date object (not pd.Timestamp)
            if isinstance(d, pd.Timestamp):
                d_py = d.date()
            else:
                d_py = d
            dt = datetime.combine(d_py, datetime.min.time()).replace(tzinfo=timezone.utc)
            try:
                sunrise = sun.get_sunrise_time(dt)
                sunset = sun.get_sunset_time(dt)
            except Exception as e:
                print(f"Failed for {city} {d_py}: {e}")
                sunrise, sunset = pd.NaT, pd.NaT
            records.append({'city_name': city, 'date': d_py, 'sunrise_time_utc': sunrise, 'sunset_time_utc': sunset})
    return pd.DataFrame(records)

def add_time_features(df, city_coords_dict, sun_table=None):
    df = df.copy()

    df['date'] = df['utc_time'].dt.date
    df['month'] = df['utc_time'].dt.month
    
    # The following commented out code calculates the sunrise and sunset time, but is slow. Instead I build a lookup table
    # using suntime as this should run much faster.

    # Prepare empty lists for sunrise/sunset etc.
    # sunrise_list, sunset_list, daylight_length_list = [], [], []
    # sunrise_frac_list, sunset_frac_list = [], []

    # Using tqdm here as this bit takes a while to run.
    # for idx, row in tqdm(df.iterrows(), total=len(df)):
    #     city = row['city_name']
    #     lat, lon = city_coords_dict[city]
    #     location = pvlib.location.Location(lat, lon, tz='UTC')  # Or use the city's local time zone if available

    #     date = row['utc_time'].date()
    #     times = pd.date_range(
    #         start=pd.Timestamp(date),
    #         end=pd.Timestamp(date) + pd.Timedelta(days=1),
    #         freq='1min', tz='UTC'
    #     )
    #     # Calculate solar position
    #     sp = location.get_solarposition(times)
    #     sunrise_time = times[sp['apparent_elevation'] > 0].min()
    #     sunset_time = times[sp['apparent_elevation'] > 0].max()

    #     sunrise_list.append(sunrise_time)
    #     sunset_list.append(sunset_time)
    #     day_length = (sunset_time - sunrise_time).total_seconds() / 3600.0
    #     daylight_length_list.append(day_length)

    #     # Sunrise/sunset as fractions of the day
    #     sunrise_frac = (sunrise_time.hour + sunrise_time.minute/60.0) / 24.0
    #     sunset_frac = (sunset_time.hour + sunset_time.minute/60.0) / 24.0
    #     sunrise_frac_list.append(sunrise_frac)
    #     sunset_frac_list.append(sunset_frac)
    
    # df['sunrise_time_utc'] = sunrise_list
    # df['sunset_time_utc'] = sunset_list
    # df['daylight_length_hr'] = daylight_length_list
    # df['sunrise_day_fraction_utc'] = sunrise_frac_list
    # df['sunset_day_fraction_utc'] = sunset_frac_list

    # If no precomputed table is given, build it now:
    if sun_table is None:
        all_dates = df['date'].unique()
        sun_table = precompute_sunrise_sunset(city_coords_dict, all_dates)

    # Merge sun_table (on city and date) into main dataframe
    df = df.merge(sun_table, on=['city_name', 'date'], how='left')

    # Remove timezone info for naive datetime columns (removes +00:00)
    df['sunrise_time_utc_naive'] = df['sunrise_time_utc'].dt.tz_localize(None)
    df['sunset_time_utc_naive'] = df['sunset_time_utc'].dt.tz_localize(None)

    # Daylight length (in hours)
    df['daylight_length_hr'] = (df['sunset_time_utc_naive'] - df['sunrise_time_utc_naive']).dt.total_seconds() / 3600.0

    neg_mask = df['daylight_length_hr'] < 0
    df.loc[neg_mask, 'daylight_length_hr'] = df.loc[neg_mask, 'daylight_length_hr'] + 24.0

    df['sunrise_time_utc'] = df['sunrise_time_utc_naive'].dt.time
    df['sunset_time_utc'] = df['sunset_time_utc_naive'].dt.time

    # Sunrise/sunset as fractions of day (UTC)
    df['sunrise_day_fraction_utc'] = (
        df['sunrise_time_utc_naive'].dt.hour / 24 +
        df['sunrise_time_utc_naive'].dt.minute / (24*60) +
        df['sunrise_time_utc_naive'].dt.second / (24*3600)
    )
    df['sunset_day_fraction_utc'] = (
        df['sunset_time_utc_naive'].dt.hour / 24 +
        df['sunset_time_utc_naive'].dt.minute / (24*60) +
        df['sunset_time_utc_naive'].dt.second / (24*3600)
    )

    df = df.drop(columns=['sunrise_time_utc_naive', 'sunset_time_utc_naive'])

    # UTC time as fraction of day
    df['utc_time_day_fraction'] = df['utc_time'].dt.hour/24.0 + df['utc_time'].dt.minute/(24.0*60) + df['utc_time'].dt.second/(24.0*3600)
    # Date as fraction of year
    df['day_of_year'] = df['utc_time'].dt.dayofyear
    df['date_fraction_of_year'] = df['day_of_year'] / 365.25

    # Day of week (Monday=0)
    df['day_of_week'] = df['utc_time'].dt.dayofweek
    # # Is weekend (0 for Mon–Fri, 1 for Sat/Sun)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Cyclical time encodings
    df['day_hour_sin'] = np.sin(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_hour_cos'] = np.cos(2 * np.pi * df['utc_time'].dt.hour / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['season'] = df['month'].apply(get_season)

    years = df['local_time'].dt.year.unique().tolist()
    es_holidays = holidays.country_holidays('ES', years=years)
    df['is_holiday'] = df['local_time'].dt.date.isin(es_holidays).astype(int)

    return df

In [ ]:
merged_power_weather_feat2 = merged_power_weather_feat.copy(deep=True)

# Precompute sunrise/sunset for all unique dates and cities
all_dates = merged_power_weather_feat2['utc_time'].dt.date.unique()
sun_table = precompute_sunrise_sunset(city_coords, all_dates)

merged_power_weather_feat2 = add_time_features(merged_power_weather_feat2, city_coords, sun_table)
#merged_power_weather_feat2 = add_time_features(merged_power_weather_feat2, city_coords)

direct_out = current_dir + '/output'
filename_out = direct_out + '/power_weather_dataset_with_new_features.csv'
merged_power_weather_feat2.to_csv(filename_out, index=False)

The following cells I check to make sure the features are computed correctly.

In [ ]:
print(merged_power_weather_feat2['utc_time'])

In [ ]:
holidays_found = (
    merged_power_weather_feat2.loc[merged_power_weather_feat2['is_holiday'] == 1, 'local_time']
    .dt.date
    .drop_duplicates()
    .sort_values()
    .tolist()
)
print("Detected holidays in dataset:")
for d in holidays_found:
    print(d)

In [ ]:
es_holiday = holidays.country_holidays('ES', years=[2015, 2016, 2017, 2018])
official = sorted(date for date, name in es_holidays.items())
print("\nOfficial Spanish national holidays between 2015 and 2018:")
for d in official:
    print(d)

In [ ]:
merged_power_weather_feat2.columns

Lastly, I will add rolling weather and power averages that I believe will improve the ML predictive modeling. Additionally, I should apply a weight to the power features as these are reported on a national level while the weather is given for only five cities. The influence of weather in a city on national power demand/generation will (to first order) scale with how much of the population lives there.

In [ ]:
merged_power_weather_feat_final = merged_power_weather_feat2.copy(deep=True)

# Weather columns to roll by city
weather_features = [
    'temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
    'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'solar_flux_Wm2'
]

# Power features to roll (national)
power_features = [
    'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
    'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation nuclear', 'generation other', 'generation other renewable', 'generation solar',
    'generation waste', 'generation wind onshore'
]

# Features to roll for both 3h and 6h (e.g., for load and price)
special_features = ['total load actual', 'price actual']

In [ ]:
# WEATHER: by city_name and sorted by utc_time
merged_power_weather_feat_final = merged_power_weather_feat_final.sort_values(['city_name', 'utc_time'])
for col in weather_features:
    # Data leakage here, don't use.
    # merged_power_weather_feat_final[f'{col}_rolling3h'] = (
    #     merged_power_weather_feat_final
    #     .sort_values(['city_name', 'utc_time'])
    #     .groupby('city_name')[col]
    #     .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    # )

    # Need to include shift(1) to ensure now future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final
        .groupby('city_name')[col]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )

    # Fill the first row of each city (where shift(1) makes it NaN) with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

# POWER: National level (no city), sorted by utc_time
merged_power_weather_feat_final = merged_power_weather_feat_final.sort_values('utc_time')
for col in power_features:
    # Data leakage
    # merged_power_weather_feat_final[f'{col}_rolling3h'] = (
    #     merged_power_weather_feat_final
    #     .sort_values('utc_time')[col]
    #     .rolling(window=3, min_periods=1)
    #     .mean()
    #     .values
    # )

    # Need to include shift(1) to ensure now future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=3, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

# Special features (total load actual, price actual): 3h and 6h
for col in special_features:
    # Data leakage
    # merged_power_weather_feat_final[f'{col}_rolling3h'] = (
    #     merged_power_weather_feat_final
    #     .sort_values('utc_time')[col]
    #     .rolling(window=3, min_periods=1)
    #     .mean()
    #     .values
    # )

    # Need to include shift(1) to ensure now future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=3, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling3h'] = (
        merged_power_weather_feat_final[f'{col}_rolling3h']
        .fillna(merged_power_weather_feat_final[col])
    )

    # Data leakage
    # merged_power_weather_feat_final[f'{col}_rolling6h'] = (
    #     merged_power_weather_feat_final
    #     .sort_values('utc_time')[col]
    #     .rolling(window=6, min_periods=1)
    #     .mean()
    #     .values
    # )

    # Need to include shift(1) to ensure now future data is leaked 
    merged_power_weather_feat_final[f'{col}_rolling6h'] = (
        merged_power_weather_feat_final[col]
        .shift(1).rolling(window=6, min_periods=1)
        .mean()
    )

    # Fill the first row of with the current value.
    merged_power_weather_feat_final[f'{col}_rolling6h'] = (
        merged_power_weather_feat_final[f'{col}_rolling6h']
        .fillna(merged_power_weather_feat_final[col])
    )

In [ ]:
time_feature_columns = [
    'date', 'month', 'sunrise_time_utc', 'sunset_time_utc', 'daylight_length_hr', 'sunrise_day_fraction_utc',
    'sunset_day_fraction_utc', 'utc_time_day_fraction', 'day_of_year', 'date_fraction_of_year',
    'day_of_week', 'is_weekend', 'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos',
    'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday'
]

# Remove from current columns and re-insert at the right spot
cols = merged_power_weather_feat_final.columns.tolist()
# Remove time_feature_columns from their current positions
cols = [c for c in cols if c not in time_feature_columns]
# Insert after 'utc_time'
utc_time_idx = cols.index('utc_time')
for offset, c in enumerate(time_feature_columns):
    cols.insert(utc_time_idx + 1 + offset, c)

In [ ]:
def insert_rolling_after_original(cols, df, rolling_suffixes=['_rolling3h', '_rolling6h']):
    new_cols = []
    already_added = set()
    for c in cols:
        new_cols.append(c)
        already_added.add(c)
        # Insert rolling variant(s) if present
        for sfx in rolling_suffixes:
            rc = c + sfx
            if rc in df.columns and rc not in already_added:
                new_cols.append(rc)
                already_added.add(rc)
    # Do not append remaining rolling columns again!
    return new_cols

# Rebuild cols to avoid duplication
cols = merged_power_weather_feat_final.columns.tolist()

# Remove the rolling columns from their current position (if present)
for col in merged_power_weather_feat_final.columns:
    if col.endswith('_rolling3h') or col.endswith('_rolling6h'):
        if col in cols:
            cols.remove(col)

# Insert rolling columns next to originals, without duplication
cols = insert_rolling_after_original(cols, merged_power_weather_feat_final)
merged_power_weather_feat_final = merged_power_weather_feat_final[cols]

In [ ]:
# 1. Main time columns
main_time_cols = ['local_time', 'utc_time', 'utc_timestamp']

# 2. Time feature columns
time_feature_cols = [
    'date', 'month', 'sunrise_time_utc', 'sunset_time_utc', 'daylight_length_hr',
    'sunrise_day_fraction_utc', 'sunset_day_fraction_utc', 'utc_time_day_fraction',
    'day_of_year', 'date_fraction_of_year', 'day_of_week', 'is_weekend', 'day_hour_sin',
    'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin',
    'day_of_year_cos', 'month_sin', 'month_cos', 'season', 'is_holiday'
]

# 3. City column
city_col = ['city_name']

# 4. The rest: everything except above (order: weather, power, forecasts, rolling, etc.)
exclude = set(main_time_cols + time_feature_cols + city_col)
rest_cols = [c for c in merged_power_weather_feat_final.columns if c not in exclude]

# To get rolling columns next to original, do this:
def get_original_and_rolling(cols):
    used = set()
    new_cols = []
    for col in cols:
        if col in used:
            continue
        new_cols.append(col)
        used.add(col)
        # Add matching rolling cols
        for suf in ['_rolling3h', '_rolling6h']:
            rolling_col = f"{col}{suf}"
            if rolling_col in cols and rolling_col not in used:
                new_cols.append(rolling_col)
                used.add(rolling_col)
    return new_cols

# Order weather and power columns with rolling features next to originals
weather_power_rolling_cols = [
    'temp', 'temp_min', 'temp_max', 'pressure', 'humidity', 'wind_speed', 'wind_deg',
    'rain_1h', 'rain_3h', 'snow_3h', 'clouds_all', 'solar_flux_Wm2', 'weather_id', 'weather_main',
    'weather_description', 'weather_icon', 'severity', 
    'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas',
    'generation fossil hard coal', 'generation fossil oil', 'generation hydro pumped storage consumption',
    'generation hydro run-of-river and poundage', 'generation hydro water reservoir',
    'generation nuclear', 'generation other', 'generation other renewable',
    'generation solar', 'generation waste', 'generation wind onshore',
    'total load actual', 'price actual'
]

# Add rolling for weather/power columns
weather_power_rolling_cols_all = []
for col in weather_power_rolling_cols:
    weather_power_rolling_cols_all.append(col)
    if f"{col}_rolling3h" in merged_power_weather_feat_final.columns:
        weather_power_rolling_cols_all.append(f"{col}_rolling3h")
    if f"{col}_rolling6h" in merged_power_weather_feat_final.columns:
        weather_power_rolling_cols_all.append(f"{col}_rolling6h")

# Other columns (forecast, weather description, etc.) in their original order
leftover_cols = [c for c in rest_cols if c not in weather_power_rolling_cols_all]

# Final column order
final_order = (
    main_time_cols +
    time_feature_cols +
    city_col +
    weather_power_rolling_cols_all +
    leftover_cols
)

# Reorder the dataframe
merged_power_weather_feat_final = merged_power_weather_feat_final[final_order]

Lastly I will implement a city weighting scheme based on the population of the five cities in this dataset based on the average between the 2011 and 2021 census data, obtained from https://www.citypopulation.de/. I've decided to manually do this for simplicity.

In [ ]:
merged_power_weather_feat_final['city_name'] = merged_power_weather_feat_final['city_name'].str.strip()

city_populations_2016 = {
    'Madrid': 3238048,
    'Barcelona': 1619286,
    'Valencia': 790448,
    'Seville': 691191,
    'Bilbao': 348553
}

total_population = sum(city_populations_2016.values())
city_weights = {city: pop / total_population for city, pop in city_populations_2016.items()}

merged_power_weather_feat_final['city_weights'] = merged_power_weather_feat_final['city_name'].map(city_weights)

### Step 9: Implement Machine Learning Models to Predict Power Generation, Total Load, & Pricing
The dataset should be ready for building different machine learning models to predict power generation (focusing on solar and wind), total load, and pricing. The ML models I will try and compare include:
* Random Forest Regression
* XGBoost
* LSTM
* Prophet
* ARIMAX (AutoRegressive Integrated Moving Average with eXogenous variables)

I will start with ARIMAX as a baseline model. The model will consist of the AutoRegressive part that uses the dependency between past observations and lagged observations; Integrated which involves differencing the time series to achieve stationarity (mean and variance are approximately constant over time); Moving Average which uses the dependency between an observation and a residual error from a moving average model applied to lagged observations; and eXogenous variables which are the external predictors (weather features) that impact the time series.

In [ ]:
# Output the current columns and datatypes so we know which ones to include in our model.
direct_out = current_dir + '/output'
filename_out = direct_out + '/power_weather_dataset_columns_datatypes.csv'
merged_power_weather_feat_final.dtypes.to_csv(filename_out, index=True)

#### ARIMAX Model to Predict Solar/Wind Power Generation, Energy Load, & Pricing

Below I implement the ARIMAX model to predict future energy generation, load, and pricing that I will then use as a baseline to compare other more sophisticated ML models.

In [ ]:
merged_power_weather_feat_final.columns

In [ ]:
# I'll start with predicting solar generation.

# --- Ensure your dataframe is sorted ---
df = merged_power_weather_feat_final.copy()
df = df.sort_values('utc_time')

target_col = 'generation solar'
past_rolling_target_feat = 'generation solar_rolling3h_past'
forecast_col = 'forecast solar day ahead'
datetime_col = 'utc_time'
city_col = 'city_name'

# Exclude categorical weather, non-numeric, and other power columns. The columns that are kept have been specified so this really
# isn't necessary, but might be useful in the future.
exclude_cols = ['local_time', 'utc_time', 'utc_timestamp', 'date', 'weather_main', 'weather_description',
                'weather_icon', 'generation solar', 'generation solar_rolling3h', 'generation wind onshore',
                'generation wind onshore_rolling3h', 'generation biomass', 'generation biomass_rolling3h',
                'generation fossil brown coal/lignite', 'generation fossil brown coal/lignite_rolling3h',
                'generation fossil gas', 'generation fossil gas_rolling3h',
                'generation fossil hard coal', 'generation fossil hard coal_rolling3h',
                'generation fossil oil', 'generation fossil oil_rolling3h',
                'generation hydro pumped storage consumption',
                'generation hydro pumped storage consumption_rolling3h',
                'generation hydro run-of-river and poundage',
                'generation hydro run-of-river and poundage_rolling3h',
                'generation hydro water reservoir',
                'generation hydro water reservoir_rolling3h', 'generation nuclear',
                'generation nuclear_rolling3h', 'generation other',
                'generation other_rolling3h', 'generation other renewable',
                'generation other renewable_rolling3h', 'generation waste',
                'generation waste_rolling3h', 'total load actual', 'total load actual_rolling3h',
                'total load actual_rolling6h', 'price actual', 'price actual_rolling3h', 'price actual_rolling6h',
                'forecast solar day ahead', 'forecast wind onshore day ahead',
                'total load forecast', 'price day ahead', 'city_name']

# Specify columns to weight based on population.
features_to_weight = ['daylight_length_hr', 'sunrise_day_fraction_utc', 'sunset_day_fraction_utc', 'temp',
                      'temp_rolling3h', 'temp_min', 'temp_min_rolling3h', 'temp_max', 'temp_max_rolling3h',
                      'pressure', 'pressure_rolling3h', 'humidity', 'humidity_rolling3h', 'wind_speed',
                      'wind_speed_rolling3h', 'wind_deg', 'wind_deg_rolling3h', 'rain_1h', 'rain_1h_rolling3h',
                      'rain_3h', 'rain_3h_rolling3h', 'snow_3h', 'snow_3h_rolling3h', 'clouds_all', 'clouds_all_rolling3h',
                      'solar_flux_Wm2', 'solar_flux_Wm2_rolling3h', 'weather_id', 'severity']

# Don't apply the city weights to these columns as they will be the same for all cities.
time_features_no_weight = ['month', 'day_of_year', 'date_fraction_of_year', 'day_of_week',
                           'is_weekend', 'season', 'is_holiday', 'utc_time_day_fraction',
                           'day_hour_sin', 'day_hour_cos', 'day_of_week_sin', 'day_of_week_cos',
                           'day_of_year_sin', 'day_of_year_cos', 'month_sin', 'month_cos']

print("Features used for ARIMAX:", features_to_weight + time_features_no_weight + [past_rolling_target_feat])

# Apply the city weights (based on population) to aggregate the weather/relevant time features across the cities.
agg_df = (
    df.groupby('utc_time', group_keys=False)
      .agg({col: lambda x: np.sum(x * df.loc[x.index, 'city_weights']) for col in features_to_weight})
)

time_feats = df.groupby('utc_time', group_keys=False)[time_features_no_weight].first()
agg_df = agg_df.join(time_feats)

# Add the national targets & forecast values
agg_df[target_col] = df.groupby('utc_time')[target_col].first()
agg_df[forecast_col] = df.groupby('utc_time')[forecast_col].first()
agg_df = agg_df.reset_index()
agg_df = agg_df.sort_values('utc_time').reset_index(drop=True)

# Add the past rolling mean of solar energy generation. We are using this instead of just the rolling mean to prevent data leakage
# into our training and test sets.
agg_df[past_rolling_target_feat] = (
    agg_df[target_col].shift(1).rolling(window=3, min_periods=1).mean()
)
# Need to fill in the first past rolling mean value with the solar generation value as it is a NaN.
agg_df.loc[0, past_rolling_target_feat] = agg_df.loc[0, target_col] 

In [ ]:
direct_out = current_dir + '/output'
filename_out = direct_out + '/agreggated_dataset_column_stats.csv'
agg_df.describe().to_csv(filename_out, index=True)

In [ ]:
def bokeh_plot_seasonal_decompose(result, title_prefix=""):
    """
    Interactive Bokeh plot of statsmodels seasonal_decompose results.
    
    Parameters:
    -----------
    result : statsmodels.tsa.seasonal.DecomposeResult
        Output of seasonal_decompose.
    title_prefix : str
        Prefix for subplot titles.

    Returns:
    --------
    plot_object : bokeh.layouts.column
        The Bokeh column layout object.
    """
    components = ['observed', 'trend', 'seasonal', 'resid']
    tooltips = [("Time", "@x{%F %H:%M}"), ("Value", "@y{0.00}")]
    formatter = {"@x": "datetime"}

    plots = []
    for comp in components:
        y = getattr(result, comp)
        p = figure(height=250, width=1200, 
                   x_axis_type='datetime',
                   tools="pan,wheel_zoom,box_zoom,reset,save",
                   title=f"{title_prefix} {comp.capitalize()}")
        p.line(y.index, y.values, line_width=2, color="royalblue")
        p.add_tools(HoverTool(
            tooltips=tooltips, formatters=formatter, mode='vline'
        ))
        p.xaxis.formatter = DatetimeTickFormatter(
            days="%d-%m-%Y",
            months="%b %Y",
            years="%Y",
            hours="%d-%m-%Y %H:%M"
        )
        p.yaxis.axis_label = comp.capitalize()
        plots.append(p)
    
    # Only show x-axis labels for bottom plot to reduce clutter
    # for plot in plots[:-1]:
    #     plot.xaxis.visible = False

    plot_object = column(*plots)

    return plot_object

In [ ]:
# Let's visualize the seasonality of the data with seasonal_decompose
from statsmodels.tsa.seasonal import seasonal_decompose

solar_series = agg_df.set_index('utc_time')['generation solar']

result = seasonal_decompose(solar_series, model='additive', period=24)  # 24 = daily cycle for hourly data

In [ ]:
# Plot the seasonality of the data.
plot_object = bokeh_plot_seasonal_decompose(result, title_prefix="Solar Generation Seaonality Plots")

show(plot_object)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Solar_seasonality.html'

title = 'Solar Power Generation Showing Seasonality using Seasonal Decompose Function'

save(plot_object, filename_out, title=title)

In [ ]:
# Ensure no NA rows in predictors or target
feature_cols = features_to_weight + time_features_no_weight
data = agg_df[[target_col, forecast_col, 'utc_time', past_rolling_target_feat] + feature_cols].dropna()
N = len(data)

#split the data into a 70/30 training and testing set.
split_idx = int(N * 0.7)

train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

train = train.copy()
test = test.copy()
train = train.set_index('utc_time', drop=False)
train = train.asfreq('h')
test = test.set_index('utc_time', drop=False)
test = test.asfreq('h')

# After split, set the past_rolling_target_feat to NaN (will be filled recursively)
test.loc[:, past_rolling_target_feat] = np.nan

# Split into X/y
y_train = train[target_col]
X_train = train[feature_cols + [past_rolling_target_feat]]
y_test = test[target_col]
X_test = test[feature_cols + [past_rolling_target_feat]]
forecast_test = test[forecast_col]
datetime_test = test['utc_time'].to_numpy(copy=True)  # Save for plotting

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Build a simple ARIMAX
# arimax_order = (1, 0, 0)  # ARIMA(p,d,q), start with (1,0,0) for a baseline
# Let's try a higher order ARIMAX model with a seasonal componenet as the baseline model performed poorly.
arimax_order = (1, 0, 1)
# ARIMA seasonal (P,D,Q,s), — seasonal AR, seasonal difference, seasonal MA, and s=seasonal period
seasonal_order = (1, 0, 1, 24)   # Will try 24 hours for seasonal period though there are weekly and monthly seasonality.
# model = SARIMAX(y_train, exog=X_train, order=arimax_order, enforce_stationarity=False, enforce_invertibility=False, freq='h')
model = SARIMAX(y_train, exog=X_train, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# # Predict on the test set
# pred = results.get_prediction(start=split_idx, end=N-1, exog=X_test)
# y_pred = pred.predicted_mean

In [ ]:
# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Check the Augmented Dickey-Fuller (ADF) unit root test and Kwiatkowski-Phillips-Schmidt-Shin test (KPSS) for whether the
# time series is stationary (important for ARIMAX modeling). If the p-value is less than 0.05 in the ADF test,
# than time series is stationary. Conversely, if the p-value is less than 0.05 for the KPSS test, then this indicates
# non-stationarity due to a trend and differencing is required.

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss

adf_result = adfuller(y_train)
print('ADF Statistic:', adf_result[0])
print('p-value of ADF:', adf_result[1])

kpss_result = kpss(y_train)
print('KPSS Statistic:', kpss_result[0])
print('p-value of KPSS:', kpss_result[1])

# Let's apply a first order differencing as the KPSS test indicates non-stationarity.
y_train_diff = y_train.diff().dropna()

print('\nTest stationarity for differenced series (order=1)')
print('--------------------------------------------------')
adf_result_diff = adfuller(y_train_diff)
print('ADF Statistic:', adf_result_diff[0])
print('p-value of ADF:', adf_result_diff[1])

kpss_result_diff = kpss(y_train_diff)
print('KPSS Statistic:', kpss_result_diff[0])
print('p-value of KPSS:', kpss_result_diff[1])

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=True)
y_train.plot(ax=axs[0], title='Original time series')
y_train_diff.plot(ax=axs[1], title='Differenced order 1');

In [ ]:
# Check ADF and KPSS if there is a seasonal drift that needs differencing (seasonal order 'D')

seasonal_diff = y_train - y_train.shift(24)  # for daily seasonality with hourly data

seasonal_diff = seasonal_diff.dropna()

inf_count = np.isinf(seasonal_diff).values.ravel().sum()
print("Number of inf values:", inf_count)

nan_count = seasonal_diff.isna().sum().sum()
print("Number of NaN values:", nan_count)

adf_result = adfuller(seasonal_diff)
print('ADF Statistic:', adf_result[0])
print('p-value of ADF:', adf_result[1])

kpss_result = kpss(seasonal_diff)
print('KPSS Statistic:', kpss_result[0])
print('p-value of KPSS:', kpss_result[1])

In [ ]:
# Check the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) for the best order of AR term p (PACF)
# and the order of MA term q (ACF).
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Autocorrelation plot for original and differentiated series
# ==============================================================================
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True)
plot_acf(y_train, ax=axs[0], lags=50, alpha=0.05)
axs[0].set_title('Autocorrelation original series')
plot_acf(seasonal_diff, ax=axs[1], lags=50, alpha=0.05)
axs[1].set_title('Autocorrelation differentiated series (order=1)');

In [ ]:
# Partial autocorrelation plot for original and differenced series
# ==============================================================================
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(6, 3), sharex=True)
plot_pacf(y_train, ax=axs[0], lags=50, alpha=0.05)
axs[0].set_title('Partial autocorrelation original series')
plot_pacf(seasonal_diff, ax=axs[1], lags=50, alpha=0.05)
axs[1].set_title('Partial autocorrelation differenced series (order=1)');
plt.tight_layout();

Based on the results of the Augmented Dickey-Fuller (ADF) unit root test and Kwiatkowski-Phillips-Schmidt-Shin test (KPSS), an order 1 differencing is needed (though the ADF and KPSS disagree on whether differencing is needed, ADF  p_value < 0.05 suggests stationarity while KPSS p_value < 0.05 suggests non-stationarity). Therefore, I will rerun the ARIMAX model with 'd' = 1.

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).

test_rec = test.copy()
rolling_vals = list(train[target_col].iloc[-3:])  # last 3 actuals values from the training data

# Place for predictions
predictions = []
rolling_means = []

for i in range(len(test_rec)):
    # Calculate rolling mean for this step (using rolling_vals, which is updated each iteration)
    curr_rolling_mean = np.mean(rolling_vals[-3:])
    test_rec.iloc[i, test_rec.columns.get_loc(past_rolling_target_feat)] = curr_rolling_mean

    # Prepare exogenous variables for this step
    X_exog = test_rec.iloc[[i]][feature_cols + [past_rolling_target_feat]].reset_index(drop=True)

    # print(X_exog.dtypes)
    # print(X_exog.head())

    # Predict with ARIMAX
    # y_pred = arimax_fit.predict(start=len(train) + i, end=len(train) + i, exog=X_exog).iloc[0]
    forecast_res = arimax_fit.get_forecast(steps=1, exog=X_exog)
    y_pred = forecast_res.predicted_mean.iloc[0]
    predictions.append(y_pred)
    rolling_means.append(curr_rolling_mean)

    # Update rolling_vals with the NEW prediction (for the next step)
    rolling_vals.append(y_pred)

In [ ]:
test_rec['ARIMAX Prediction'] = predictions
test_rec['generation solar_rolling3h_past'] = rolling_means  # Updated rolling means

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred = np.full(N, np.nan)
full_pred[split_idx:] = test_rec['ARIMAX Prediction']  # y_pred is from your ARIMAX model's prediction
roll_means = np.full(N, np.nan)
roll_means[:split_idx] = train[past_rolling_target_feat]
roll_means[split_idx:] = test_rec[past_rolling_target_feat]

plot_df = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred,
    past_rolling_target_feat: roll_means,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

for col in feature_cols:
    plot_df[col] = data[col].to_numpy(copy=True)

In [ ]:
# Create a Bokeh plotting function to plot the actual, ML predicted, and forecast power generation
def ML_future_energy_predict(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                             features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black',
                             color_features='black', labels=None, features_labels=None, symbols=None, symbols_features=None,
                             test_pred_col=None, lower_conf_col=None, upper_conf_col=None):

    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}
    
    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        p.xaxis.major_label_orientation = 120
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        if normalize:
            max_primary_data = dataframe[column].max()
        else:
            max_primary_data = 1

        # Only show the first column by default, hide others
        visible = True if i == 0 else False

        # Scatter plot with symbols.
        if symbols:
            p.scatter(dataframe[x_axis_column], dataframe[column]/max_primary_data, size=10,
                      marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                      legend_label=label, visible=visible)
            
        # Add a line to connect the symbols
        p.line(dataframe[x_axis_column], dataframe[column]/max_primary_data,
               line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label,
               visible=visible)

        marker_index = marker_index + 1
        color_index = color_index + 1
        label_index = label_index + 1

    if test_pred_col is not None and lower_conf_col is not None and upper_conf_col is not None:
        mask = ~dataframe[lower_conf_col].isna()
        if normalize:
            mask_primary = ~dataframe[test_pred_col].isna()
            max_primary_data = dataframe[test_pred_col][mask_primary].max()
        else:
            max_primary_data = 1
        p.varea(x=dataframe[x_axis_column][mask],
                y1=dataframe[lower_conf_col][mask]/max_primary_data,
                y2=dataframe[upper_conf_col][mask]/max_primary_data,
                fill_color="grey", fill_alpha=0.3,
                legend_label="ARIMAX 1sigma CI")

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            if normalize:
                max_features_data = dataframe[feature_column].max()
            else:
                max_features_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 else False

            # Scatter plot with symbols. Only plot the energy usage as a function of time.
            if symbols_features:
                p.scatter(dataframe[x_axis_column], dataframe[feature_column]/max_features_data,
                          y_range_name="features", size=10, marker=symbols_features[marker_index],
                          color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label,
                          visible=visible)
            # Add a line to connect the symbols
            p.line(dataframe[x_axis_column], dataframe[feature_column]/max_features_data, y_range_name="features",
                   line_width=2, color=color_plot_values[color_index], alpha=0.5,
                   legend_label=feature_label, visible=visible)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

In [ ]:
# Call the plotting function
p = ML_future_energy_predict(plot_df, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'], features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'blue'],
                             color_features=['yellow'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'diamond'],
                             symbols_features=['circle'])

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_seasonality.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
# Run ARIMAX a second time but with 'd' = 1 in the arimax_order to apply a first order differencing.
arimax_order = (1, 1, 1)
# ARIMA seasonal (P,D,Q,s), — seasonal AR, seasonal difference, seasonal MA, and s=seasonal period
seasonal_order = (1, 0, 1, 24)   # Will try 24 hours for seasonal period though there are weekly and monthly seasonality.
# model = SARIMAX(y_train, exog=X_train, order=arimax_order, enforce_stationarity=False, enforce_invertibility=False, freq='h')
model = SARIMAX(y_train, exog=X_train, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

In [ ]:
# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_w_differencing.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).

test_rec = test.copy()
rolling_vals = list(train[target_col].iloc[-3:])  # last 3 actuals values from the training data

# Place for predictions
predictions = []
rolling_means = []

for i in range(len(test_rec)):
    # Calculate rolling mean for this step (using rolling_vals, which is updated each iteration)
    curr_rolling_mean = np.mean(rolling_vals[-3:])
    test_rec.iloc[i, test_rec.columns.get_loc(past_rolling_target_feat)] = curr_rolling_mean

    # Prepare exogenous variables for this step
    X_exog = test_rec.iloc[[i]][feature_cols + [past_rolling_target_feat]].reset_index(drop=True)

    # print(X_exog.dtypes)
    # print(X_exog.head())

    # Predict with ARIMAX
    # y_pred = arimax_fit.predict(start=len(train) + i, end=len(train) + i, exog=X_exog).iloc[0]
    forecast_res = arimax_fit.get_forecast(steps=1, exog=X_exog)
    y_pred = forecast_res.predicted_mean.iloc[0]
    predictions.append(y_pred)
    rolling_means.append(curr_rolling_mean)

    # Update rolling_vals with the NEW prediction (for the next step)
    rolling_vals.append(y_pred)

In [ ]:
test_rec['ARIMAX Prediction'] = predictions
test_rec['generation solar_rolling3h_past'] = rolling_means  # Updated rolling means

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred = np.full(N, np.nan)
full_pred[split_idx:] = test_rec['ARIMAX Prediction']  # y_pred is from your ARIMAX model's prediction
roll_means = np.full(N, np.nan)
roll_means[:split_idx] = train[past_rolling_target_feat]
roll_means[split_idx:] = test_rec[past_rolling_target_feat]

plot_df = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred,
    past_rolling_target_feat: roll_means,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

for col in feature_cols:
    plot_df[col] = data[col].to_numpy(copy=True)

In [ ]:
# Call the plotting function
p = ML_future_energy_predict(plot_df, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'], features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'blue'],
                             color_features=['yellow'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'diamond'],
                             symbols_features=['circle'])

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_seasonality_differencing.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

Now let's try to find the best SARIMAX orders using a grid search method using the strongest features.

In [ ]:
params = arimax_fit.params
pvalues = arimax_fit.pvalues

# Create DataFrame for inspection
coef_table = pd.DataFrame({
    'coef': params,
    'pvalue': pvalues
})

# Only keep exogenous columns
exog_cols = [col for col in coef_table.index if col in X_train.columns or col == 'generation solar_rolling3h_past']

# Filter by p-value
signif_coef_table = coef_table.loc[exog_cols]
signif_coef_table = signif_coef_table[signif_coef_table['pvalue'] <= 0.1]

print("Significant exogenous features (p <= 0.1):")
print(signif_coef_table)
signif_cols = signif_coef_table.index.tolist()

In [ ]:
X_train_sig = X_train[signif_cols]
X_test_sig  = X_test[signif_cols]

In [ ]:
# Due to the length of the dataset and how long it takes to run the ARIMAX model, I will reduce the length of the
# training dataset used to determine the best order coefficients to the first 6 months. Additionally, I will downsample the data
# to three hour time intervals to improve speed in the grid search.
first_date = X_train_sig.index[0]
cutoff_date = first_date + pd.DateOffset(months=6)

mask = (X_train_sig.index >= first_date) & (X_train_sig.index < cutoff_date)
X_train_sig_6mo = X_train_sig.loc[mask]
y_train_6mo = y_train.loc[mask]

# 2. Downsample both X and y to 3-hour intervals using median
X_train_sig_6mo_ds = X_train_sig_6mo.resample('3H').median().dropna()
y_train_6mo_ds = y_train_6mo.resample('3H').median().dropna()

# 3. Make sure indices match after resampling (may have missing y or X)
# (optional, but safest—ensures alignment)
common_idx = X_train_sig_6mo_ds.index.intersection(y_train_6mo_ds.index)
X_train_sig_6mo_ds = X_train_sig_6mo_ds.loc[common_idx]
y_train_6mo_ds = y_train_6mo_ds.loc[common_idx]

print(X_train_sig.index[0])
print(X_train_sig.index[-1])

print(X_train_sig_6mo.index[0])
print(X_train_sig_6mo.index[-1])

print(y_train_6mo.index[0])
print(y_train_6mo.index[-1])

print(f"Original rows: {len(X_train_sig_6mo)}, Downsampled rows: {len(X_train_sig_6mo_ds)}")

In [ ]:
from itertools import product
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")


# Parameter grid
p = d = q = range(0, 4)
P = D = Q = range(0, 4)
d = D = [0, 1]
s = 24

# All combinations, but restrict d and D to [0, 1] if desired
order_list = list(product(p, d, q))
seasonal_order_list = list(product(P, D, Q))

# param_grid = [
#     (order, (seasonal[0], seasonal[1], seasonal[2], s))
#     for order in order_list
#     for seasonal in seasonal_order_list
# ]

# Define worker function at the top level
def fit_sarimax(order, seasonal, y, X, s):
    try:
        mod = SARIMAX(y, exog=X, order=order,
                      seasonal_order=(seasonal[0], seasonal[1], seasonal[2], s),
                      enforce_stationarity=False, enforce_invertibility=False)
        res = mod.fit(disp=False)
        return (order, seasonal, res.aic, res.bic, None)
    except Exception as e:
        return (order, seasonal, np.inf, np.inf, str(e))

# Set number of parallel jobs (leave one core free)
n_jobs = max(1, os.cpu_count() - 1)

# Create the parameter grid
params = [(order, seasonal) for order in order_list for seasonal in seasonal_order_list]

# Run parallel grid search with progress bar
results = Parallel(n_jobs=n_jobs)(
    delayed(fit_sarimax)(order, seasonal, y_train_6mo_ds, X_train_sig_6mo_ds, s)
    for order, seasonal in tqdm(params)
)

# Convert results to DataFrame for easy analysis
results_df = pd.DataFrame(results, columns=['order', 'seasonal', 'aic', 'bic', 'error'])

# Filter out failed fits (AIC==np.inf)
valid_results = results_df[results_df['aic'] != np.inf]

# Best models
best_aic_row = valid_results.loc[valid_results['aic'].idxmin()]
best_bic_row = valid_results.loc[valid_results['bic'].idxmin()]

print("Best AIC:", best_aic_row['aic'])
print("Best AIC order:", best_aic_row['order'], "seasonal:", best_aic_row['seasonal'])

print("Best BIC:", best_bic_row['bic'])
print("Best BIC order:", best_bic_row['order'], "seasonal:", best_bic_row['seasonal'])

# Optionally, review any errors
if not results_df['error'].isnull().all():
    print("Some combinations failed. Here's a sample of errors:")
    print(results_df[results_df['error'].notnull()].head())

In [ ]:
# The results from the AIC and BIC are:
# Best AIC: 20103.645171484328
# Best AIC order: (2, 0, 3) seasonal: (1, 1, 3)
# Best BIC: 20259.85873717056
# Best BIC order: (2, 0, 3) seasonal: (1, 1, 3)

# Given that q and Q values are at the upper end of the explored range from the grid search, I will expand the search for q and Q
# values while keeping the other values fixed.

# Parameter grid
p = [best_aic_row['order'][0]]
d = [best_aic_row['order'][1]]
q_range = [3, 4, 5]
P = [best_aic_row['seasonal'][0]]
D = [best_aic_row['seasonal'][1]]
Q_range = [3, 4, 5]
s = 24

order_list_new = list(product(p, d, q_range))
seasonal_order_list_new = list(product(P, D, Q_range))

# Create the parameter grid
params_new = [(order, seasonal) for order in order_list_new for seasonal in seasonal_order_list_new]

# Run parallel grid search with progress bar
new_results = Parallel(n_jobs=n_jobs)(
    delayed(fit_sarimax)(order, seasonal, y_train_6mo_ds, X_train_sig_6mo_ds, s)
    for order, seasonal in tqdm(params_new)
)

# Convert results to DataFrame for easy analysis
results_new_df = pd.DataFrame(new_results, columns=['order', 'seasonal', 'aic', 'bic', 'error'])

# Filter out failed fits (AIC==np.inf)
valid_results_new = results_new_df[results_new_df['aic'] != np.inf]

# Best models
best_aic_row_new = valid_results_new.loc[valid_results_new['aic'].idxmin()]
best_bic_row_new = valid_results_new.loc[valid_results_new['bic'].idxmin()]

print("Best AIC:", best_aic_row_new['aic'])
print("Best AIC order:", best_aic_row_new['order'], "seasonal:", best_aic_row_new['seasonal'])

print("Best BIC:", best_bic_row_new['bic'])
print("Best BIC order:", best_bic_row_new['order'], "seasonal:", best_bic_row_new['seasonal'])

# Optionally, review any errors
if not results_new_df['error'].isnull().all():
    print("Some combinations failed. Here's a sample of errors:")
    print(results_new_df[results_new_df['error'].notnull()].head())

In [ ]:
results_new_df = pd.DataFrame(new_results, columns=['order', 'seasonal', 'aic', 'bic', 'error'])

# Filter out failed fits (AIC==np.inf)
valid_results_new = results_new_df[results_new_df['aic'] != np.inf]

# Best models
best_aic_row_new = valid_results_new.loc[valid_results_new['aic'].idxmin()]
best_bic_row_new = valid_results_new.loc[valid_results_new['bic'].idxmin()]

print("Best AIC:", best_aic_row_new['aic'])
print("Best AIC order:", best_aic_row_new['order'], "seasonal:", best_aic_row_new['seasonal'])

print("Best BIC:", best_bic_row_new['bic'])
print("Best BIC order:", best_bic_row_new['order'], "seasonal:", best_bic_row_new['seasonal'])

# Optionally, review any errors
if not results_new_df['error'].isnull().all():
    print("Some combinations failed. Here's a sample of errors:")
    print(results_new_df[results_new_df['error'].notnull()].head())

In [ ]:
# Run ARIMAX again using best order and seasonal coefficients with the most significant exogenous features
# Best AIC: 19321.240593927767
# Best AIC order: (2, 0, 5) seasonal: (1, 1, 5)
# Best BIC: 19496.998494494568
# Best BIC order: (2, 0, 5) seasonal: (1, 1, 5)

arimax_order = (best_aic_row_new['order'][0], best_aic_row_new['order'][1], best_aic_row_new['order'][2])
seasonal_order = (best_bic_row_new['seasonal'][0], best_bic_row_new['seasonal'][1], best_bic_row_new['seasonal'][2], 24)

model = SARIMAX(y_train, exog=X_train_sig, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_w_best_coefficients.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).
import copy

test_rec = test.copy()
rolling_vals = list(train[target_col].iloc[-3:])  # last 3 actuals values from the training data

# Place for predictions
predictions = []
rolling_means = []

signif_cols = signif_coef_table.index.tolist()

for i in range(len(test_rec)):
    # Calculate rolling mean for this step (using rolling_vals, which is updated each iteration)
    curr_rolling_mean = np.mean(rolling_vals[-3:])
    test_rec.iloc[i, test_rec.columns.get_loc(past_rolling_target_feat)] = curr_rolling_mean

    # Prepare exogenous variables for this step
    X_exog = test_rec.iloc[[i]][signif_cols].reset_index(drop=True)

    # print(X_exog.dtypes)
    # print(X_exog.head())

    # Predict with ARIMAX
    # y_pred = arimax_fit.predict(start=len(train) + i, end=len(train) + i, exog=X_exog).iloc[0]
    forecast_res = arimax_fit.get_forecast(steps=1, exog=X_exog)
    y_pred = forecast_res.predicted_mean.iloc[0]
    predictions.append(y_pred)
    rolling_means.append(curr_rolling_mean)

    # Update rolling_vals with the NEW prediction (for the next step)
    rolling_vals.append(y_pred)
    
test_rec['ARIMAX Prediction'] = predictions
test_rec['generation solar_rolling3h_past'] = rolling_means  # Updated rolling means

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred_new = np.full(N, np.nan)
full_pred_new[split_idx:] = test_rec['ARIMAX Prediction']  # y_pred is from your ARIMAX model's prediction
roll_means_new = np.full(N, np.nan)
roll_means_new[:split_idx] = train[past_rolling_target_feat]
roll_means_new[split_idx:] = test_rec[past_rolling_target_feat]

plot_df_new = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred_new,
    past_rolling_target_feat: roll_means_new,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

signif_cols2 = copy.deepcopy(signif_cols)
signif_cols2.remove('generation solar_rolling3h_past')

for col in signif_cols2:
    plot_df_new[col] = data[col].to_numpy(copy=True)

In [ ]:
# Call the plotting function
p = ML_future_energy_predict(plot_df_new, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'], features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'blue'],
                             color_features=['yellow'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'diamond'],
                             symbols_features=['circle'])

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_best_coefficients.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
# Run ARIMAX again using best order and seasonal coefficients with the most significant exogenous features
# Best AIC: 19321.240593927767
# Best AIC order: (2, 0, 5) seasonal: (1, 1, 5)
# Best BIC: 19496.998494494568
# Best BIC order: (2, 0, 5) seasonal: (1, 1, 5)

# Try q order of 3

arimax_order = (best_aic_row_new['order'][0], best_aic_row_new['order'][1], 3)
seasonal_order = (best_bic_row_new['seasonal'][0], best_bic_row_new['seasonal'][1], 3, 24)

model = SARIMAX(y_train, exog=X_train_sig, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_w_q_coef_3.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).
import copy

test_rec = test.copy()
rolling_vals = list(train[target_col].iloc[-3:])  # last 3 actuals values from the training data

# Place for predictions
predictions = []
rolling_means = []

signif_cols = signif_coef_table.index.tolist()

for i in range(len(test_rec)):
    # Calculate rolling mean for this step (using rolling_vals, which is updated each iteration)
    curr_rolling_mean = np.mean(rolling_vals[-3:])
    test_rec.iloc[i, test_rec.columns.get_loc(past_rolling_target_feat)] = curr_rolling_mean

    # Prepare exogenous variables for this step
    X_exog = test_rec.iloc[[i]][signif_cols].reset_index(drop=True)

    # print(X_exog.dtypes)
    # print(X_exog.head())

    # Predict with ARIMAX
    # y_pred = arimax_fit.predict(start=len(train) + i, end=len(train) + i, exog=X_exog).iloc[0]
    forecast_res = arimax_fit.get_forecast(steps=1, exog=X_exog)
    y_pred = forecast_res.predicted_mean.iloc[0]
    predictions.append(y_pred)
    rolling_means.append(curr_rolling_mean)

    # Update rolling_vals with the NEW prediction (for the next step)
    rolling_vals.append(y_pred)
    
test_rec['ARIMAX Prediction'] = predictions
test_rec['generation solar_rolling3h_past'] = rolling_means  # Updated rolling means

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred_new = np.full(N, np.nan)
full_pred_new[split_idx:] = test_rec['ARIMAX Prediction']  # y_pred is from your ARIMAX model's prediction
roll_means_new = np.full(N, np.nan)
roll_means_new[:split_idx] = train[past_rolling_target_feat]
roll_means_new[split_idx:] = test_rec[past_rolling_target_feat]

plot_df_new = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred_new,
    past_rolling_target_feat: roll_means_new,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

signif_cols2 = copy.deepcopy(signif_cols)
signif_cols2.remove('generation solar_rolling3h_past')

for col in signif_cols2:
    plot_df_new[col] = data[col].to_numpy(copy=True)

In [ ]:
# Call the plotting function
p = ML_future_energy_predict(plot_df_new, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'], features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'blue'],
                             color_features=['yellow'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'diamond'],
                             symbols_features=['circle'])

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_q_coef_3.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
# Run ARIMAX again using best order and seasonal coefficients with the most significant exogenous features
# Best AIC: 19321.240593927767
# Best AIC order: (2, 0, 5) seasonal: (1, 1, 5)
# Best BIC: 19496.998494494568
# Best BIC order: (2, 0, 5) seasonal: (1, 1, 5)

# Try q order of 3
# try D of 0

arimax_order = (2, 0, 3)
seasonal_order = (2, 0, 3, 24)

model = SARIMAX(y_train, exog=X_train_sig, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_w_q_coef_3.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).
import copy

test_rec = test.copy()
rolling_vals = list(train[target_col].iloc[-3:])  # last 3 actuals values from the training data

# Place for predictions
predictions = []
rolling_means = []

signif_cols = signif_coef_table.index.tolist()

for i in range(len(test_rec)):
    # Calculate rolling mean for this step (using rolling_vals, which is updated each iteration)
    curr_rolling_mean = np.mean(rolling_vals[-3:])
    test_rec.iloc[i, test_rec.columns.get_loc(past_rolling_target_feat)] = curr_rolling_mean

    # Prepare exogenous variables for this step
    X_exog = test_rec.iloc[[i]][signif_cols].reset_index(drop=True)

    # print(X_exog.dtypes)
    # print(X_exog.head())

    # Predict with ARIMAX
    # y_pred = arimax_fit.predict(start=len(train) + i, end=len(train) + i, exog=X_exog).iloc[0]
    forecast_res = arimax_fit.get_forecast(steps=1, exog=X_exog)
    y_pred = forecast_res.predicted_mean.iloc[0]
    predictions.append(y_pred)
    rolling_means.append(curr_rolling_mean)

    # Update rolling_vals with the NEW prediction (for the next step)
    rolling_vals.append(y_pred)
    
test_rec['ARIMAX Prediction'] = predictions
test_rec['generation solar_rolling3h_past'] = rolling_means  # Updated rolling means

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred_new = np.full(N, np.nan)
full_pred_new[split_idx:] = test_rec['ARIMAX Prediction']  # y_pred is from your ARIMAX model's prediction
roll_means_new = np.full(N, np.nan)
roll_means_new[:split_idx] = train[past_rolling_target_feat]
roll_means_new[split_idx:] = test_rec[past_rolling_target_feat]
train_pred_new = np.full(N, np.nan)
train_pred_new[:split_idx] = arimax_fit.fittedvalues

plot_df_new = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred_new,
    past_rolling_target_feat: roll_means_new,
    'ARIMAX Test Prediction': train_pred_new,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

signif_cols2 = copy.deepcopy(signif_cols)
signif_cols2.remove('generation solar_rolling3h_past')

for col in signif_cols2:
    plot_df_new[col] = data[col].to_numpy(copy=True)

In [ ]:
# Call the plotting function
p = ML_future_energy_predict(plot_df_new, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 
                                                       'ARIMAX Test Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Test Prediction,'
                             + 'Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['generation solar_rolling3h_past'],
                             features_ylabel=[r"$$\mathrm{Rolling\ Solar\ Generation\ (MWh)}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'black', 'blue'],
                             color_features=['green'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power',
                                     'ARIMAX Predicted Test Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Rolling Solar Generation'], symbols=['star', 'triangle', 'square', 'diamond'],
                             symbols_features=['circle'])

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_q_coef_3.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
# Let's plot residuals of predictions - actual solar power generation from both the training and testing sets
# to get a better understand of what is going on, look for trends, etc. Additionally, I will plot a histogram
# of the residuals to see if they are skewed.

from bokeh.plotting import figure, show
from bokeh.layouts import row, column
from bokeh.models import Span, Legend
import numpy as np
import pandas as pd

def plot_train_test_residuals_bokeh(y_train, arimax_fit, y_test, test_pred, test_index, title="Train/Test Residuals Analysis"):
    # Calculate residuals
    resid_train = y_train - arimax_fit.fittedvalues
    resid_test = y_test - test_pred

    # If your index isn't datetime, use integer index for plotting
    if not np.issubdtype(y_train.index.dtype, np.datetime64):
        x_train = np.arange(len(y_train))
        x_test = test_index if np.issubdtype(type(test_index[0]), np.datetime64) else np.arange(len(y_test))
        x_axis_type = None
    else:
        x_train = y_train.index
        x_test = test_index
        x_axis_type = "datetime"
    
    # Time Series Residual Plot
    p = figure(title=title, height=800, width=1200, x_axis_type=x_axis_type, tools="pan,wheel_zoom,box_zoom,reset,save")
    l1 = p.line(x_train, resid_train, color="navy", alpha=0.6, legend_label="Train Residuals", line_width=2)
    l2 = p.line(x_test, resid_test, color="firebrick", alpha=0.8, legend_label="Test Residuals", line_width=2)
    p.add_layout(Span(location=0, dimension='width', line_color='black', line_dash='dashed'))
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    p.xaxis.axis_label = "Datetime"
    p.yaxis.axis_label = r"$$\mathrm{Power\ Generation\ Residuals\ (Actual - Predicted, MWh)}$$"
    # Format the x-axis datetime labels.
    p.xaxis.formatter = DatetimeTickFormatter(
        minutes="%d-%m-%y %H:%M",
        hours="%d-%m-%y %H:%M",
        days="%d-%m-%y %H:%M",
        months="%d-%m-%y %H:%M",
        years="%d-%m-%y %H:%M"
    )
    
    # Residual Histogram
    # Use pandas dropna to avoid plotting nans (can happen if short train/test)
    train_resid_nonan = resid_train.dropna()
    test_resid_nonan = pd.Series(resid_test).dropna()
    hist1, edges1 = np.histogram(train_resid_nonan, bins=50)
    hist2, edges2 = np.histogram(test_resid_nonan, bins=50)
    
    p_hist = figure(title="Residuals Histogram for Solar Power Generation", height=800, width=1200)
    p_hist.quad(top=hist1, bottom=0, left=edges1[:-1], right=edges1[1:], fill_color="navy", line_color="white",
                alpha=0.6, legend_label="Train")
    p_hist.quad(top=hist2, bottom=0, left=edges2[:-1], right=edges2[1:], fill_color="firebrick", line_color="white",
                alpha=0.4, legend_label="Test")
    p_hist.legend.location = "top_left"
    p_hist.legend.click_policy = "hide"
    p_hist.xaxis.axis_label = r"$$\mathrm{Power\ Generation\ Residuals\ (Actual - Predicted, MWh)}$$"
    p_hist.yaxis.axis_label = "Count"

    return p, p_hist

In [ ]:
p, p_hist = plot_train_test_residuals_bokeh(y_train, arimax_fit, y_test, test_rec['ARIMAX Prediction'], test_rec.index,
                                            title="Train/Test Residuals Analysis for Solar Power Generation")

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_with_q_coef_3.html'

title = 'Train/Test Residuals Analysis for Solar Power Generation'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_with_q_coef_3.html'

title = 'Histogram of the Residuals for Solar Power Generation'

save(p_hist, filename_out, title=title)

In [ ]:
true_rolling = (
    pd.concat([train[target_col], test[target_col]])
    .rolling(window=3, min_periods=1)
    .mean()
    .iloc[len(train):]
    .values
)

num = 100

plt.figure(figsize=(12,5))
plt.plot(test_rec['utc_time'][:num], rolling_means[:num], label="Predicted Rolling Mean (used in model)")
plt.plot(test_rec['utc_time'][:num], true_rolling[:num], label="True Rolling Mean (not seen by model)")
plt.legend()
plt.title("Predicted vs. True 3-hour Rolling Mean in Test Set")
plt.show()

In [ ]:
# Run ARIMAX but without the 3 hour rolling power feature
# Best AIC: 19321.240593927767
# Best AIC order: (2, 0, 5) seasonal: (1, 1, 5)
# Best BIC: 19496.998494494568
# Best BIC order: (2, 0, 5) seasonal: (1, 1, 5)

# Try q order of 3
# try D of 0

arimax_order = (2, 0, 3)
seasonal_order = (2, 0, 3, 24)

X_train_sig_no_roll = X_train_sig.copy(deep=True)
X_train_sig_no_roll.drop('generation solar_rolling3h_past', axis=1, inplace=True)

model = SARIMAX(y_train, exog=X_train_sig_no_roll, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_w_q_coef_3_no_roll.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Predict future solar energy generation. This will be done recursively as the generation solar_rolling3h_past
# feature must be populated with the predicted values (not the actual values from generation solar that we are
# trying to predict as this would result in data leakage).
test_rec = test.copy()

X_test_sig_no_roll = X_test_sig.copy(deep=True)
X_test_sig_no_roll.drop('generation solar_rolling3h_past', axis=1, inplace=True)

signif_cols = signif_coef_table.index.tolist()

signif_cols_no_roll = [col for col in signif_cols if col != 'generation solar_rolling3h_past']

forecast_res = arimax_fit.get_forecast(steps=len(X_test_sig_no_roll), exog=X_test_sig_no_roll)
y_pred_test = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.32)  # 1 Sigma confidence interval
y_pred_test.index = X_test_sig_no_roll.index  # Align index, if needed

test_rec['ARIMAX Prediction'] = y_pred_test

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred_new = np.full(N, np.nan)
full_pred_new[split_idx:] = test_rec['ARIMAX Prediction']
train_pred_new = np.full(N, np.nan)
train_pred_new[:split_idx] = arimax_fit.fittedvalues

plot_df_new = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred_new,
    'ARIMAX Test Prediction': train_pred_new,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

plot_df_new['ARIMAX Lower Bound'] = np.nan
plot_df_new['ARIMAX Upper Bound'] = np.nan
plot_df_new.loc[split_idx:, 'ARIMAX Lower Bound'] = conf_int.iloc[:, 0].values
plot_df_new.loc[split_idx:, 'ARIMAX Upper Bound'] = conf_int.iloc[:, 1].values

# Add significant cols (excluding rolling mean)
for col in signif_cols_no_roll:
    plot_df_new[col] = data[col].to_numpy(copy=True)

In [ ]:
p = ML_future_energy_predict(plot_df_new, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 
                                                       'ARIMAX Test Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Test Prediction,'
                             + ' Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'],
                             features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'black', 'blue'],
                             color_features=['green'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power',
                                     'ARIMAX Predicted Test Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'square', 'diamond'],
                             symbols_features=['circle'], test_pred_col='ARIMAX Test Prediction',
                             lower_conf_col='ARIMAX Lower Bound', upper_conf_col='ARIMAX Upper Bound')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_with_q_coef_3_no_roll.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
p, p_hist = plot_train_test_residuals_bokeh(y_train, arimax_fit, y_test, test_rec['ARIMAX Prediction'], test_rec.index,
                                            title="Train/Test Residuals Analysis for Solar Power Generation")

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_with_q_coef_3_no_roll.html'

title = 'Train/Test Residuals Analysis for Solar Power Generation'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_with_q_coef_3_no_roll.html'

title = 'Histogram of the Residuals for Solar Power Generation'

save(p_hist, filename_out, title=title)

In [ ]:
# I have determined that the 'generation solar_rolling3h_past' feature is causing issues with the ARIMAX model
# so I have decided to remove it as a feature. I have added back a few other features that I think might have
# some influence on the predictive power of the model.

columns_use = ['daylight_length_hr', 'sunrise_day_fraction_utc',
               'sunset_day_fraction_utc', 'temp', 'temp_rolling3h', 'temp_min',
               'temp_min_rolling3h', 'temp_max', 'temp_max_rolling3h', 'pressure',
               'pressure_rolling3h', 'humidity', 'humidity_rolling3h', 'wind_speed',
               'wind_speed_rolling3h', 'wind_deg', 'wind_deg_rolling3h', 'rain_1h',
               'rain_1h_rolling3h', 'rain_3h', 'rain_3h_rolling3h', 'snow_3h',
               'snow_3h_rolling3h', 'clouds_all', 'clouds_all_rolling3h',
               'solar_flux_Wm2', 'solar_flux_Wm2_rolling3h', 'weather_id', 'severity',
               'month', 'day_of_year', 'date_fraction_of_year', 'utc_time_day_fraction',
               'day_hour_sin', 'day_hour_cos', 'day_of_year_sin', 'day_of_year_cos',
               'month_sin', 'month_cos']

X_train_sig2 = X_train[columns_use]
X_test_sig2  = X_test[columns_use]

first_date = X_train_sig2.index[0]
cutoff_date = first_date + pd.DateOffset(months=6)

mask = (X_train_sig2.index >= first_date) & (X_train_sig2.index < cutoff_date)
X_train_sig_6mo = X_train_sig2.loc[mask]
y_train_6mo = y_train.loc[mask]

# 2. Downsample both X and y to 3-hour intervals using median
X_train_sig_6mo_ds = X_train_sig_6mo.resample('3H').median().dropna()
y_train_6mo_ds = y_train_6mo.resample('3H').median().dropna()

# 3. Make sure indices match after resampling (may have missing y or X)
# (optional, but safest—ensures alignment)
common_idx = X_train_sig_6mo_ds.index.intersection(y_train_6mo_ds.index)
X_train_sig_6mo_ds = X_train_sig_6mo_ds.loc[common_idx]
y_train_6mo_ds = y_train_6mo_ds.loc[common_idx]

# Parameter grid
p = d = q = range(0, 4)
P = D = Q = range(0, 4)
d = D = [0, 1]
s = 24

# All combinations, but restrict d and D to [0, 1] if desired
order_list = list(product(p, d, q))
seasonal_order_list = list(product(P, D, Q))

# Set number of parallel jobs (leave one core free)
n_jobs = max(1, os.cpu_count() - 1)

# Create the parameter grid
params = [(order, seasonal) for order in order_list for seasonal in seasonal_order_list]

In [ ]:
# Run parallel grid search with progress bar
results = Parallel(n_jobs=n_jobs)(
    delayed(fit_sarimax)(order, seasonal, y_train_6mo_ds, X_train_sig_6mo_ds, s)
    for order, seasonal in tqdm(params)
)

# Convert results to DataFrame for easy analysis
results_df = pd.DataFrame(results, columns=['order', 'seasonal', 'aic', 'bic', 'error'])

# Filter out failed fits (AIC==np.inf)
valid_results = results_df[results_df['aic'] != np.inf]

# Best models
best_aic_row = valid_results.loc[valid_results['aic'].idxmin()]
best_bic_row = valid_results.loc[valid_results['bic'].idxmin()]

print("Best AIC:", best_aic_row['aic'])
print("Best AIC order:", best_aic_row['order'], "seasonal:", best_aic_row['seasonal'])

print("Best BIC:", best_bic_row['bic'])
print("Best BIC order:", best_bic_row['order'], "seasonal:", best_bic_row['seasonal'])

# Optionally, review any errors
if not results_df['error'].isnull().all():
    print("Some combinations failed. Here's a sample of errors:")
    print(results_df[results_df['error'].notnull()].head())

In [ ]:
# arimax_order = (best_aic_row['order'][0], best_aic_row['order'][1], best_aic_row['order'][2])
# seasonal_order = (best_bic_row['seasonal'][0], best_bic_row['seasonal'][1], best_bic_row['seasonal'][2], 24)

arimax_order = (1, 1, 2)
seasonal_order = (1, 1, 2, 24)

model = SARIMAX(y_train, exog=X_train_sig2, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/ARIMAX_model_train_fit_results_reasonable_coefficients.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
test_rec = test.copy()

forecast_res = arimax_fit.get_forecast(steps=len(X_test_sig2), exog=X_test_sig2)
y_pred_test = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.32)  # 1 Sigma confidence interval
y_pred_test.index = X_test_sig2.index  # Align index, if needed

test_rec['ARIMAX Prediction'] = y_pred_test
test_rec['Forecast Solar Day Ahead'] = data.loc[split_idx:, forecast_col].to_numpy(copy=True)

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred = np.full(N, np.nan)
full_pred[split_idx:] = test_rec['ARIMAX Prediction']
train_pred = np.full(N, np.nan)
train_pred[:split_idx] = arimax_fit.fittedvalues

plot_df = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Solar': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction': full_pred,
    'ARIMAX Train Prediction': train_pred,
    'Forecast Solar Day Ahead': data[forecast_col].to_numpy(copy=True)
})

plot_df['ARIMAX Lower Bound'] = np.nan
plot_df['ARIMAX Upper Bound'] = np.nan
plot_df.loc[split_idx:, 'ARIMAX Lower Bound'] = conf_int.iloc[:, 0].values
plot_df.loc[split_idx:, 'ARIMAX Upper Bound'] = conf_int.iloc[:, 1].values

# Add significant cols (excluding rolling mean)
for col in signif_cols_no_roll:
    plot_df[col] = data[col].to_numpy(copy=True)

In [ ]:
p = ML_future_energy_predict(plot_df, 'utc_time', ['Actual Solar', 'ARIMAX Prediction', 
                                                       'ARIMAX Train Prediction', 'Forecast Solar Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Solar Power Generation, ARIMAX Prediction, Train Prediction,'
                             + ' Forecast Day Ahead vs UTC Time and Solar Flux',
                             feature_columns=['solar_flux_Wm2'],
                             features_ylabel=[r"$$\mathrm{Solar\ Flux\ (W\ m^{-2})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'black', 'blue'],
                             color_features=['green'],
                             labels=['Actual Solar Power Generation', 'ARIMAX Predicted Solar Power',
                                     'ARIMAX Predicted Train Solar Power', 'Forecast Solar Day Ahead'],
                             features_labels=['Solar Flux'], symbols=['star', 'triangle', 'square', 'diamond'],
                             symbols_features=['circle'], test_pred_col='ARIMAX Train Prediction',
                             lower_conf_col='ARIMAX Lower Bound', upper_conf_col='ARIMAX Upper Bound')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_solar_power_ARIMAX_reasonable_coefficients.html'

title = 'Solar Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Solar Flux'

save(p, filename_out, title=title)

In [ ]:
p, p_hist = plot_train_test_residuals_bokeh(y_train, arimax_fit, y_test, test_rec['ARIMAX Prediction'], test_rec.index,
                                            title="Train/Test Residuals Analysis for Solar Power Generation")

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_reasonable_coefficients.html'

title = 'Train/Test Residuals Analysis for Solar Power Generation'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_reasonable_coefficients.html'

title = 'Histogram of the Residuals for Solar Power Generation'

save(p_hist, filename_out, title=title)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    nonzero = y_true != 0
    return np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100

# --- Full Test Set Metrics ---
metrics = {}

# Full range
y_true_full = y_test.values
y_arimax_full = test_rec['ARIMAX Prediction'].values
y_forecast_full = test_rec['Forecast Solar Day Ahead'].values

metrics['All Test Data'] = {
    'ARIMAX': {
        "MSE": mean_squared_error(y_true_full, y_arimax_full),
        "RMSE": np.sqrt(mean_squared_error(y_true_full, y_arimax_full)),
        "MAE": mean_absolute_error(y_true_full, y_arimax_full),
        "MAPE": mean_absolute_percentage_error(y_true_full, y_arimax_full),
        "R2": r2_score(y_true_full, y_arimax_full)
    },
    'Forecast': {
        "MSE": mean_squared_error(y_true_full, y_forecast_full),
        "RMSE": np.sqrt(mean_squared_error(y_true_full, y_forecast_full)),
        "MAE": mean_absolute_error(y_true_full, y_forecast_full),
        "MAPE": mean_absolute_percentage_error(y_true_full, y_forecast_full),
        "R2": r2_score(y_true_full, y_forecast_full)
    }
}

# --- First Day Only (first 24 points/24 hours) ---
y_true_day1 = y_test.values[:24]
y_arimax_day1 = test_rec['ARIMAX Prediction'].values[:24]
y_forecast_day1 = test_rec['Forecast Solar Day Ahead'].values[:24]

metrics['First 24 Hours'] = {
    'ARIMAX': {
        "MSE": mean_squared_error(y_true_day1, y_arimax_day1),
        "RMSE": np.sqrt(mean_squared_error(y_true_day1, y_arimax_day1)),
        "MAE": mean_absolute_error(y_true_day1, y_arimax_day1),
        "MAPE": mean_absolute_percentage_error(y_true_day1, y_arimax_day1),
        "R2": r2_score(y_true_day1, y_arimax_day1)
    },
    'Forecast': {
        "MSE": mean_squared_error(y_true_day1, y_forecast_day1),
        "RMSE": np.sqrt(mean_squared_error(y_true_day1, y_forecast_day1)),
        "MAE": mean_absolute_error(y_true_day1, y_forecast_day1),
        "MAPE": mean_absolute_percentage_error(y_true_day1, y_forecast_day1),
        "R2": r2_score(y_true_day1, y_forecast_day1)
    }
}

# --- Format as a Pretty DataFrame ---
df_metrics = pd.DataFrame({
    (section, model): metrics[section][model]
    for section in metrics.keys()
    for model in metrics[section].keys()
})

# Reorder for readability
df_metrics = df_metrics[[
    ('All Test Data', 'ARIMAX'), ('All Test Data', 'Forecast'),
    ('First 24 Hours', 'ARIMAX'), ('First 24 Hours', 'Forecast')
]]

# Print to console
print(df_metrics.round(3))

# Output to CSV and TXT (tabular)
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/arimax_vs_forecast_metrics_reasonable_coefficients.csv'
df_metrics.to_csv(filename_out)

# with open('arimax_vs_forecast_metrics.txt', 'w') as f:
#     f.write(df_metrics.round(3).to_string())

In [ ]:
# Now we will run a benchmark SARIMAX model to predict wind generation.

df = merged_power_weather_feat_final.copy()
df = df.sort_values('utc_time')

target_col = 'generation wind onshore'
# past_rolling_target_feat = 'generation solar_rolling3h_past'
forecast_col = 'forecast wind onshore day ahead'
datetime_col = 'utc_time'
city_col = 'city_name'

# Exclude categorical weather, non-numeric, and other power columns. The columns that are kept have been specified so this really
# isn't necessary, but might be useful in the future.
exclude_cols = ['local_time', 'utc_time', 'utc_timestamp', 'date', 'weather_main', 'weather_description',
                'weather_icon', 'generation solar', 'generation solar_rolling3h', 'generation wind onshore',
                'generation wind onshore_rolling3h', 'generation biomass', 'generation biomass_rolling3h',
                'generation fossil brown coal/lignite', 'generation fossil brown coal/lignite_rolling3h',
                'generation fossil gas', 'generation fossil gas_rolling3h',
                'generation fossil hard coal', 'generation fossil hard coal_rolling3h',
                'generation fossil oil', 'generation fossil oil_rolling3h',
                'generation hydro pumped storage consumption',
                'generation hydro pumped storage consumption_rolling3h',
                'generation hydro run-of-river and poundage',
                'generation hydro run-of-river and poundage_rolling3h',
                'generation hydro water reservoir',
                'generation hydro water reservoir_rolling3h', 'generation nuclear',
                'generation nuclear_rolling3h', 'generation other',
                'generation other_rolling3h', 'generation other renewable',
                'generation other renewable_rolling3h', 'generation waste',
                'generation waste_rolling3h', 'total load actual', 'total load actual_rolling3h',
                'total load actual_rolling6h', 'price actual', 'price actual_rolling3h', 'price actual_rolling6h',
                'forecast solar day ahead', 'forecast wind onshore day ahead',
                'total load forecast', 'price day ahead', 'city_name']

# Specify columns to weight based on population.
features_to_weight = ['daylight_length_hr', 'sunrise_day_fraction_utc', 'sunset_day_fraction_utc', 'temp',
                      'temp_rolling3h', 'temp_min', 'temp_min_rolling3h', 'temp_max', 'temp_max_rolling3h',
                      'pressure', 'pressure_rolling3h', 'humidity', 'humidity_rolling3h', 'wind_speed',
                      'wind_speed_rolling3h', 'wind_deg', 'wind_deg_rolling3h', 'rain_1h', 'rain_1h_rolling3h',
                      'rain_3h', 'rain_3h_rolling3h', 'snow_3h', 'snow_3h_rolling3h', 'clouds_all', 'clouds_all_rolling3h',
                      'solar_flux_Wm2', 'solar_flux_Wm2_rolling3h', 'weather_id', 'severity']

# Don't apply the city weights to these columns as they will be the same for all cities.
time_features_no_weight = ['month', 'day_of_year', 'date_fraction_of_year', 'utc_time_day_fraction',
                           'day_hour_sin', 'day_hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'month_sin',
                           'month_cos']

print("Features used for ARIMAX:", features_to_weight + time_features_no_weight)

# Apply the city weights (based on population) to aggregate the weather/relevant time features across the cities.
agg_df = (
    df.groupby('utc_time', group_keys=False)
      .agg({col: lambda x: np.sum(x * df.loc[x.index, 'city_weights']) for col in features_to_weight})
)

time_feats = df.groupby('utc_time', group_keys=False)[time_features_no_weight].first()
agg_df = agg_df.join(time_feats)

# Add the national targets & forecast values
agg_df[target_col] = df.groupby('utc_time')[target_col].first()
agg_df[forecast_col] = df.groupby('utc_time')[forecast_col].first()
agg_df = agg_df.reset_index()
agg_df = agg_df.sort_values('utc_time').reset_index(drop=True)

# # Add the past rolling mean of solar energy generation. We are using this instead of just the rolling mean to prevent data leakage
# # into our training and test sets.
# agg_df[past_rolling_target_feat] = (
#     agg_df[target_col].shift(1).rolling(window=3, min_periods=1).mean()
# )
# # Need to fill in the first past rolling mean value with the solar generation value as it is a NaN.
# agg_df.loc[0, past_rolling_target_feat] = agg_df.loc[0, target_col] 

In [ ]:
# Let's visualize the seasonality of the data with seasonal_decompose
from statsmodels.tsa.seasonal import seasonal_decompose

wind_series = agg_df.set_index('utc_time')['generation wind onshore']

for period in [24, 48, 72, 168]:
    result = seasonal_decompose(wind_series, model='additive', period=period)

    # Plot the seasonality of the data.
    plot_object = bokeh_plot_seasonal_decompose(result, title_prefix="Wind Generation Seaonality Plots with Period " + str(period)
                                                + " hrs")

    # Save plots.
    direct_out = current_dir + '/output/results/plots/'
    
    # Create output directory if it doesn't already exist
    if not os.path.exists(direct_out):
        os.makedirs(direct_out)
    
    filename_out = direct_out + '/wind_seasonality_period_' + str(period) + 'hrs.html'
    
    title = 'Wind Power Generation Showing Seasonality using Seasonal Decompose Function with Period ' + str(period) + ' hrs'
    
    save(plot_object, filename_out, title=title)

In [ ]:
# Let's visualize the periodicity of the data using a periodogram via fast fourier transform (FFT)
# since it is not obvious what the periodicity of the wind generation should be.

N = len(wind_series)
dt = 1  # hourly spacing
freqs = np.fft.fftfreq(N, d=dt)
fft_vals = np.fft.fft(wind_series.values)
amplitude = np.abs(fft_vals)

# Only look at positive frequencies
pos_mask = freqs > 0
periods = 1 / freqs[pos_mask]  # period in hours

p = figure(width=1200, height=800, x_axis_type="log",
           title="FFT Periodogram of Wind Power",
           tools="pan,wheel_zoom,box_zoom,reset,save,hover")

p.line(periods, amplitude[pos_mask], line_width=2)
p.xaxis.axis_label = "Period (hours, log scale)"
p.yaxis.axis_label = "Amplitude"
#p.y_range = Range1d(0, np.percentile(amplitude[pos_mask], 99.9))
filename_out = direct_out + '/Periodogram_wind_power_FFT.html'
    
title = 'Wind Power Generation Periodogram using FFT'

save(p, filename_out, title=title)

In [ ]:
# Ensure no NA rows in predictors or target
feature_cols = features_to_weight + time_features_no_weight
data = agg_df[[target_col, forecast_col, 'utc_time'] + feature_cols].dropna()
N = len(data)

#split the data into a 70/30 training and testing set.
split_idx = int(N * 0.7)

train = data.iloc[:split_idx]
test = data.iloc[split_idx:]

train = train.copy()
test = test.copy()
train = train.set_index('utc_time', drop=False)
train = train.asfreq('h')
test = test.set_index('utc_time', drop=False)
test = test.asfreq('h')

# Split into X/y
y_train = train[target_col]
X_train = train[feature_cols]
y_test = test[target_col]
X_test = test[feature_cols]
forecast_test = test[forecast_col]
datetime_test = test['utc_time'].to_numpy(copy=True)  # Save for plotting

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Build a simple ARIMAX
# arimax_order = (1, 0, 0)  # ARIMA(p,d,q), start with (1,0,0) for a baseline
# Let's try a higher order ARIMAX model with a seasonal componenet as the baseline model performed poorly.
arimax_order = (1, 0, 1)
# ARIMA seasonal (P,D,Q,s), — seasonal AR, seasonal difference, seasonal MA, and s=seasonal period
seasonal_order = (1, 0, 1, 24)   # Will try 24 hours for seasonal period though there are weekly and monthly seasonality.
model = SARIMAX(y_train, exog=X_train, order=arimax_order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False, freq='h')

# Fit the model
arimax_fit = model.fit(disp=False)
print(arimax_fit.summary())

# Output model fit stats to file
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/SARIMAX_model_train_fit_results_wind.txt'
with open(filename_out, 'w') as f:
    f.write(str(arimax_fit.summary()))

In [ ]:
# Check the Augmented Dickey-Fuller (ADF) unit root test and Kwiatkowski-Phillips-Schmidt-Shin test (KPSS) for whether the
# time series is stationary (important for ARIMAX modeling). If the p-value is less than 0.05 in the ADF test,
# than time series is stationary. Conversely, if the p-value is less than 0.05 for the KPSS test, then this indicates
# non-stationarity due to a trend and differencing is required.

adf_result = adfuller(y_train)
print('ADF Statistic:', adf_result[0])
print('p-value of ADF:', adf_result[1])

kpss_result = kpss(y_train)
print('KPSS Statistic:', kpss_result[0])
print('p-value of KPSS:', kpss_result[1])

# Let's apply a first order differencing as the KPSS test indicates non-stationarity.
y_train_diff = y_train.diff().dropna()

print('\nTest stationarity for differenced series (order=1)')
print('--------------------------------------------------')
adf_result_diff = adfuller(y_train_diff)
print('ADF Statistic:', adf_result_diff[0])
print('p-value of ADF:', adf_result_diff[1])

kpss_result_diff = kpss(y_train_diff)
print('KPSS Statistic:', kpss_result_diff[0])
print('p-value of KPSS:', kpss_result_diff[1])

In [ ]:
# It appears some differencing is required based on the KPSS statistic. I will output the prediction results
# without differencing and then compare with using differencing.

# Predict future wind energy generation.
test_rec = test.copy()

forecast_res_wind = arimax_fit.get_forecast(steps=len(X_test), exog=X_test)
y_pred_test = forecast_res_wind.predicted_mean
conf_int = forecast_res_wind.conf_int(alpha=0.32)  # 1 Sigma confidence interval
y_pred_test.index = X_test.index  # Align index, if needed

test_rec['ARIMAX Prediction Wind'] = y_pred_test
test_rec['Forecast Wind Day Ahead'] = data.loc[split_idx:, forecast_col].to_numpy(copy=True)

# Create new dataframe to plot the ARIMAX results.
# For plotting full range with NaN for prediction in training range
full_pred = np.full(N, np.nan)
full_pred[split_idx:] = test_rec['ARIMAX Prediction Wind']
train_pred = np.full(N, np.nan)
train_pred[:split_idx] = arimax_fit.fittedvalues

plot_df = pd.DataFrame({
    'utc_time': data['utc_time'].to_numpy(copy=True),
    'Actual Wind': data[target_col].to_numpy(copy=True),
    'ARIMAX Prediction Wind': full_pred,
    'ARIMAX Train Prediction Wind': train_pred,
    'Forecast Wind Day Ahead': data[forecast_col].to_numpy(copy=True)
})

plot_df['ARIMAX Lower Bound'] = np.nan
plot_df['ARIMAX Upper Bound'] = np.nan
plot_df.loc[split_idx:, 'ARIMAX Lower Bound'] = conf_int.iloc[:, 0].values
plot_df.loc[split_idx:, 'ARIMAX Upper Bound'] = conf_int.iloc[:, 1].values

# Add significant cols
for col in feature_cols:
    plot_df[col] = data[col].to_numpy(copy=True)

In [ ]:
p = ML_future_energy_predict(plot_df, 'utc_time', ['Actual Wind', 'ARIMAX Prediction Wind', 
                                                       'ARIMAX Train Prediction Wind', 'Forecast Wind Day Ahead'],
                             r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Power\ Generation\ (MWh)}$$",
                             'Wind Power Generation, ARIMAX Prediction Wind, Train Prediction Wind,'
                             + ' Forecast Day Ahead vs UTC Time and Wind Speed',
                             feature_columns=['wind_speed'],
                             features_ylabel=[r"$$\mathrm{Wind\ Speed\ (m\ s^{-1})}$$"],
                             p=None, normalize=False, other_colors=False, color_plot=['dorange', 'red', 'black', 'blue'],
                             color_features=['green'],
                             labels=['Actual Wind Power Generation', 'ARIMAX Predicted Wind Power',
                                     'ARIMAX Predicted Train Wind Power', 'Forecast Wind Day Ahead'],
                             features_labels=['Wind Speed'], symbols=['star', 'triangle', 'square', 'diamond'],
                             symbols_features=['circle'], test_pred_col='ARIMAX Prediction Wind',
                             lower_conf_col='ARIMAX Lower Bound', upper_conf_col='ARIMAX Upper Bound')

show(p)

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Predicted_wind_power_ARIMAX_initial_coefficients.html'

title = 'Wind Power Generation, ARIMAX Prediction, Forecast Day Ahead vs UTC Time and Wind Speed'

save(p, filename_out, title=title)

In [ ]:
p, p_hist = plot_train_test_residuals_bokeh(y_train, arimax_fit, y_test, test_rec['ARIMAX Prediction Wind'], test_rec.index,
                                            title="Train/Test Residuals Analysis for Wind Power Generation")

# Save plots.
direct_out = current_dir + '/output/results/plots/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Residuals_actual_predicted_train_test_wind_power.html'

title = 'Train/Test Residuals Analysis for Wind Power Generation'

save(p, filename_out, title=title)

filename_out = direct_out + '/Histogram_residuals_train_test_wind_power.html'

title = 'Histogram of the Residuals for Wind Power Generation'

save(p_hist, filename_out, title=title)

In [ ]:
# --- Full Test Set Metrics ---
metrics = {}

# Full range
y_true_full = y_test.values
y_arimax_full = test_rec['ARIMAX Prediction Wind'].values
y_forecast_full = test_rec['Forecast Wind Day Ahead'].values

metrics['All Test Data'] = {
    'ARIMAX': {
        "MSE": mean_squared_error(y_true_full, y_arimax_full),
        "RMSE": np.sqrt(mean_squared_error(y_true_full, y_arimax_full)),
        "MAE": mean_absolute_error(y_true_full, y_arimax_full),
        "MAPE": mean_absolute_percentage_error(y_true_full, y_arimax_full),
        "R2": r2_score(y_true_full, y_arimax_full)
    },
    'Forecast': {
        "MSE": mean_squared_error(y_true_full, y_forecast_full),
        "RMSE": np.sqrt(mean_squared_error(y_true_full, y_forecast_full)),
        "MAE": mean_absolute_error(y_true_full, y_forecast_full),
        "MAPE": mean_absolute_percentage_error(y_true_full, y_forecast_full),
        "R2": r2_score(y_true_full, y_forecast_full)
    }
}

# --- First Day Only (first 24 points/24 hours) ---
y_true_day1 = y_test.values[:24]
y_arimax_day1 = test_rec['ARIMAX Prediction Wind'].values[:24]
y_forecast_day1 = test_rec['Forecast Wind Day Ahead'].values[:24]

metrics['First 24 Hours'] = {
    'ARIMAX': {
        "MSE": mean_squared_error(y_true_day1, y_arimax_day1),
        "RMSE": np.sqrt(mean_squared_error(y_true_day1, y_arimax_day1)),
        "MAE": mean_absolute_error(y_true_day1, y_arimax_day1),
        "MAPE": mean_absolute_percentage_error(y_true_day1, y_arimax_day1),
        "R2": r2_score(y_true_day1, y_arimax_day1)
    },
    'Forecast': {
        "MSE": mean_squared_error(y_true_day1, y_forecast_day1),
        "RMSE": np.sqrt(mean_squared_error(y_true_day1, y_forecast_day1)),
        "MAE": mean_absolute_error(y_true_day1, y_forecast_day1),
        "MAPE": mean_absolute_percentage_error(y_true_day1, y_forecast_day1),
        "R2": r2_score(y_true_day1, y_forecast_day1)
    }
}

# --- Format as a Pretty DataFrame ---
df_metrics = pd.DataFrame({
    (section, model): metrics[section][model]
    for section in metrics.keys()
    for model in metrics[section].keys()
})

# Reorder for readability
df_metrics = df_metrics[[
    ('All Test Data', 'ARIMAX'), ('All Test Data', 'Forecast'),
    ('First 24 Hours', 'ARIMAX'), ('First 24 Hours', 'Forecast')
]]

# Print to console
print(df_metrics.round(3))

# Output to CSV and TXT (tabular)
direct_out = current_dir + '/output/results'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/arimax_vs_forecast_metrics_wind_power.csv'
df_metrics.to_csv(filename_out)